# Global Wheat Detection — fill-in baseline

Problem family: **Object detection**. Official metric: **Mean per-image TP/(TP+FP+FN), IoU .50
to .75**.

**Learning objective:** Scratch Faster R-CNN with MobileNet FPN.

[Dataset and original task](https://www.kaggle.com/competitions/global-wheat-detection) ·
[Primary solution evidence](https://github.com/dungnb1333/Global-Wheat-Dection-1st-Place)

The linked first-place repository includes Faster R-CNN FPN alongside EfficientDet, fusion and
extra data. This baseline uses one MobileNet-FPN detector; the advanced companion uses one
ResNet-FPN detector. Neither adds external data, test pseudo-labels or the winning ensemble.

**Watch for:** Split images, not box rows. Torchvision consumes xyxy; Kaggle submits
confidence,x,y,width,height. This is not COCO mAP.

**Standalone:** this file contains every project-specific helper it uses. Only installed
third-party libraries and your local data/assets are required; no repository checkout, source
imports, project installation or external bootstrap is used. Only the optional dataset cell
needs internet when enabled; all training works offline. Edit DATA_DIR and
output/storage settings in stage 1. Train media on local disk, then persist your output bundle.

This notebook is a learning exercise, not a claimed leaderboard reproduction. No trained
weights or real Kaggle data are bundled. Keep grading answers closed. Use only explicitly
permitted resources. Internet/prepared-code access is not assumed on contest day.

Complete the named exercises and their nearby checks in order. Setup and safety helpers are
supplied.

After the lesson, close both notebooks and rebuild the essential route from memory. See the
repository's blank-notebook playbook for the final transfer test.

## How to use this lesson

Complete one named exercise, run its nearby checks, then interpret the result. Supplied cells
contain setup or integration; edit only configuration or marked exercise bodies. Every baseline
code cell retains a collapsed full reference immediately afterward. Use Hint 1, then Hint 2,
before opening a reference.

By the end you should be able to state the data/metric contract, choose a legal split, construct
one correct model input, validate a saved candidate and deliver an ID-aligned file. Mini-checks
use tiny separate examples or the current training batch; they do not certify real-data quality.

 | Stage | Goal |
|---|---|
| 1 | Environment and configuration |
| 1a | Optional dataset acquisition |
| 2 | Understand and load the data contract |
| 3 | Inspect examples and distributions |
| 4 | Build and test the validation split |
| 5 | Measure imbalance and choose a response |
| 6 | Build features or one correct batch |
| 7 | Connect outputs, loss and official metric |
| 8 | Train and preserve a recoverable candidate |
| 9 | Interpret validation and compare experiments |
| 10 | Reload and infer every test item |
| 11 | Deliver and verify the submission |



## 1. Environment and configuration

Run the supplied imports and runtime helpers in order. Edit paths, time budget and model policy
in the configuration cells. Inspect the actual device; CPU checks cannot establish GPU
throughput. Supplied storage/safety definitions are folded where the viewer supports it: run
them, but reconstruct the short algorithmic functions during closed-book practice.



In [ ]:
import json
import pickle
import random
import shutil
import sys
import time
import zipfile
from collections.abc import Iterable, Mapping, Sequence
from dataclasses import asdict, dataclass, field, is_dataclass
from pathlib import Path
from time import perf_counter
from types import SimpleNamespace
from typing import Any
from uuid import uuid4

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from numpy.typing import ArrayLike
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader

# ruff: noqa: E402



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>import json
import pickle
import random
import shutil
import sys
import time
import zipfile
from collections.abc import Iterable, Mapping, Sequence
from dataclasses import asdict, dataclass, field, is_dataclass
from pathlib import Path
from time import perf_counter
from types import SimpleNamespace
from typing import Any
from uuid import uuid4

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from numpy.typing import ArrayLike
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader

# ruff: noqa: E402

</code></pre>
</details>


### Supplied helper — `CompetitionConfig`

Explicit notebook state; no installed project is required.




In [ ]:
@dataclass
class CompetitionConfig:
    """Explicit notebook state; no installed project is required."""

    data_dir: Path
    output_dir: Path
    seed: int = 42
    num_workers: int = 0
    pretrained_allowed: bool = False
    time_budget_minutes: float = 60
    device: str = "auto"

    def prepare(self) -> None:
        self.device = (
            ("cuda" if torch.cuda.is_available() else "cpu")
            if self.device == "auto"
            else self.device
        )
        self.output_dir.mkdir(parents=True, exist_ok=True)
        random.seed(self.seed)
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.seed)

    def to_dict(self) -> dict:
        return {**asdict(self), "data_dir": str(self.data_dir), "output_dir": str(self.output_dir)}




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>@dataclass
class CompetitionConfig:
    &quot;&quot;&quot;Explicit notebook state; no installed project is required.&quot;&quot;&quot;

    data_dir: Path
    output_dir: Path
    seed: int = 42
    num_workers: int = 0
    pretrained_allowed: bool = False
    time_budget_minutes: float = 60
    device: str = &quot;auto&quot;

    def prepare(self) -&gt; None:
        self.device = (
            (&quot;cuda&quot; if torch.cuda.is_available() else &quot;cpu&quot;)
            if self.device == &quot;auto&quot;
            else self.device
        )
        self.output_dir.mkdir(parents=True, exist_ok=True)
        random.seed(self.seed)
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.seed)

    def to_dict(self) -&gt; dict:
        return {**asdict(self), &quot;data_dir&quot;: str(self.data_dir), &quot;output_dir&quot;: str(self.output_dir)}


</code></pre>
</details>


### Supplied helper — `TimeBudget`

Leave delivery time available; a running batch/fit cannot be interrupted safely.




In [ ]:
class TimeBudget:
    """Leave delivery time available; a running batch/fit cannot be interrupted safely."""

    def __init__(self, minutes: float, reserve_minutes: float = 10) -> None:
        if not 0 <= reserve_minutes < minutes:
            raise ValueError("Require 0 <= reserve < total minutes")
        self.deadline = time.monotonic() + 60 * (minutes - reserve_minutes)

    @property
    def should_stop(self) -> bool:
        return time.monotonic() >= self.deadline

    def checkpoint(self, stage: str) -> None:
        if self.should_stop:
            raise TimeoutError(f"Delivery reserve reached before {stage}")




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>class TimeBudget:
    &quot;&quot;&quot;Leave delivery time available; a running batch/fit cannot be interrupted safely.&quot;&quot;&quot;

    def __init__(self, minutes: float, reserve_minutes: float = 10) -&gt; None:
        if not 0 &lt;= reserve_minutes &lt; minutes:
            raise ValueError(&quot;Require 0 &lt;= reserve &lt; total minutes&quot;)
        self.deadline = time.monotonic() + 60 * (minutes - reserve_minutes)

    @property
    def should_stop(self) -&gt; bool:
        return time.monotonic() &gt;= self.deadline

    def checkpoint(self, stage: str) -&gt; None:
        if self.should_stop:
            raise TimeoutError(f&quot;Delivery reserve reached before {stage}&quot;)


</code></pre>
</details>


### Supplied helper — `runtime_paths`

Media stays on local disk; optional durable storage is an explicit destination.




In [ ]:
def runtime_paths(
    *, runtime_dir: Path | None = None, persistent_dir: Path | None = None
) -> SimpleNamespace:
    """Media stays on local disk; optional durable storage is an explicit destination."""
    root = (
        Path(runtime_dir)
        if runtime_dir is not None
        else (Path("/content/olp_runtime") if "google.colab" in sys.modules else Path.cwd())
    )
    data, output = root / "data", root / "outputs"
    data.mkdir(parents=True, exist_ok=True)
    output.mkdir(parents=True, exist_ok=True)
    return SimpleNamespace(
        data_dir=data,
        output_dir=output,
        persistent_dir=Path(persistent_dir) if persistent_dir is not None else None,
    )




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def runtime_paths(
    *, runtime_dir: Path | None = None, persistent_dir: Path | None = None
) -&gt; SimpleNamespace:
    &quot;&quot;&quot;Media stays on local disk; optional durable storage is an explicit destination.&quot;&quot;&quot;
    root = (
        Path(runtime_dir)
        if runtime_dir is not None
        else (Path(&quot;/content/olp_runtime&quot;) if &quot;google.colab&quot; in sys.modules else Path.cwd())
    )
    data, output = root / &quot;data&quot;, root / &quot;outputs&quot;
    data.mkdir(parents=True, exist_ok=True)
    output.mkdir(parents=True, exist_ok=True)
    return SimpleNamespace(
        data_dir=data,
        output_dir=output,
        persistent_dir=Path(persistent_dir) if persistent_dir is not None else None,
    )


</code></pre>
</details>


### Supplied helper — `mount_google_drive`

Call only when an explicitly configured Colab path uses Drive.




In [ ]:
def mount_google_drive() -> None:
    """Call only when an explicitly configured Colab path uses Drive."""
    from google.colab import drive

    drive.mount("/content/drive")




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def mount_google_drive() -&gt; None:
    &quot;&quot;&quot;Call only when an explicitly configured Colab path uses Drive.&quot;&quot;&quot;
    from google.colab import drive

    drive.mount(&quot;/content/drive&quot;)


</code></pre>
</details>


### Supplied helper — `stage_data`

Copy a folder or extract a ZIP once, then train from the local destination.




In [ ]:
def stage_data(source: Path | str, destination: Path | str) -> Path:
    """Copy a folder or extract a ZIP once, then train from the local destination."""
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if source == destination or destination.is_relative_to(source):
        raise ValueError("Staging destination must be separate from its source")
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError("Reuse staged DATA_DIR and set DATA_ARCHIVE=None")
    destination.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        with zipfile.ZipFile(source) as archive:
            for member in archive.infolist():
                if not (destination / member.filename).resolve().is_relative_to(destination):
                    raise ValueError("Archive contains a path outside the destination")
            archive.extractall(destination)
    return destination




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def stage_data(source: Path | str, destination: Path | str) -&gt; Path:
    &quot;&quot;&quot;Copy a folder or extract a ZIP once, then train from the local destination.&quot;&quot;&quot;
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if source == destination or destination.is_relative_to(source):
        raise ValueError(&quot;Staging destination must be separate from its source&quot;)
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError(&quot;Reuse staged DATA_DIR and set DATA_ARCHIVE=None&quot;)
    destination.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        with zipfile.ZipFile(source) as archive:
            for member in archive.infolist():
                if not (destination / member.filename).resolve().is_relative_to(destination):
                    raise ValueError(&quot;Archive contains a path outside the destination&quot;)
            archive.extractall(destination)
    return destination


</code></pre>
</details>


### Supplied helper — `gpu_report`

Inspect the actual runtime before choosing image size, batch size or precision.




In [ ]:
def gpu_report() -> dict:
    """Inspect the actual runtime before choosing image size, batch size or precision."""
    if not torch.cuda.is_available():
        return {"device": "cpu", "precision": "float32"}
    free, total = torch.cuda.mem_get_info()
    return {
        "device": torch.cuda.get_device_name(),
        "free_GiB": free / 2**30,
        "total_GiB": total / 2**30,
        "precision": "bf16" if torch.cuda.is_bf16_supported() else "fp16",
    }




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def gpu_report() -&gt; dict:
    &quot;&quot;&quot;Inspect the actual runtime before choosing image size, batch size or precision.&quot;&quot;&quot;
    if not torch.cuda.is_available():
        return {&quot;device&quot;: &quot;cpu&quot;, &quot;precision&quot;: &quot;float32&quot;}
    free, total = torch.cuda.mem_get_info()
    return {
        &quot;device&quot;: torch.cuda.get_device_name(),
        &quot;free_GiB&quot;: free / 2**30,
        &quot;total_GiB&quot;: total / 2**30,
        &quot;precision&quot;: &quot;bf16&quot; if torch.cuda.is_bf16_supported() else &quot;fp16&quot;,
    }


</code></pre>
</details>


### Configure this run

Edit these values before proceeding. DATA_DIR points at your staged pack; choose a separate
empty folder for an online download. Keep model permission explicit, start with a small row/time
budget, and let the actual device report guide neural batch size. Each run writes to a new
candidate directory.



In [ ]:
FAMILY, VARIANT = "06_wheat", "baseline"
DATA_ARCHIVE, PERSISTENT_DIR = None, None
if "google.colab" in sys.modules and any(
    "/content/drive/" in str(value) for value in (DATA_ARCHIVE, PERSISTENT_DIR)
):
    mount_google_drive()
paths = runtime_paths(persistent_dir=PERSISTENT_DIR)
DATA_DIR = (
    paths.data_dir / "learning" / FAMILY
    if "google.colab" in sys.modules
    else Path.cwd() / "data" / "learning" / FAMILY
)
if DATA_ARCHIVE is not None:
    stage_data(DATA_ARCHIVE, DATA_DIR)
RUN_ID = f"{FAMILY}-{VARIANT}-{uuid4().hex[:8]}"
OUTPUT = paths.output_dir / "learning" / RUN_ID
DURABLE_OUTPUT = paths.persistent_dir / RUN_ID if paths.persistent_dir is not None else None
PRETRAINED_ALLOWED, WEIGHT_PATH = False, None
SEED, MAX_ROWS, IMAGE_SIZE, BATCH_SIZE, EPOCHS = 42, 300, 384, 2, 2
TIME_BUDGET_MINUTES, DELIVERY_RESERVE_MINUTES = 60, 10
GROUP_COLUMN, TIME_COLUMN = "source", None
USE_IMBALANCE_WEIGHTS = False
MODEL_NAME = "fasterrcnn_mobilenet_v3_large_fpn"
IMAGE_SUFFIX = ".jpg"



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>FAMILY, VARIANT = &quot;06_wheat&quot;, &quot;baseline&quot;
DATA_ARCHIVE, PERSISTENT_DIR = None, None
if &quot;google.colab&quot; in sys.modules and any(
    &quot;/content/drive/&quot; in str(value) for value in (DATA_ARCHIVE, PERSISTENT_DIR)
):
    mount_google_drive()
paths = runtime_paths(persistent_dir=PERSISTENT_DIR)
DATA_DIR = (
    paths.data_dir / &quot;learning&quot; / FAMILY
    if &quot;google.colab&quot; in sys.modules
    else Path.cwd() / &quot;data&quot; / &quot;learning&quot; / FAMILY
)
if DATA_ARCHIVE is not None:
    stage_data(DATA_ARCHIVE, DATA_DIR)
RUN_ID = f&quot;{FAMILY}-{VARIANT}-{uuid4().hex[:8]}&quot;
OUTPUT = paths.output_dir / &quot;learning&quot; / RUN_ID
DURABLE_OUTPUT = paths.persistent_dir / RUN_ID if paths.persistent_dir is not None else None
PRETRAINED_ALLOWED, WEIGHT_PATH = False, None
SEED, MAX_ROWS, IMAGE_SIZE, BATCH_SIZE, EPOCHS = 42, 300, 384, 2, 2
TIME_BUDGET_MINUTES, DELIVERY_RESERVE_MINUTES = 60, 10
GROUP_COLUMN, TIME_COLUMN = &quot;source&quot;, None
USE_IMBALANCE_WEIGHTS = False
MODEL_NAME = &quot;fasterrcnn_mobilenet_v3_large_fpn&quot;
IMAGE_SUFFIX = &quot;.jpg&quot;

</code></pre>
</details>


### Configure this run

Edit these values before proceeding. DATA_DIR points at your staged pack; choose a separate
empty folder for an online download. Keep model permission explicit, start with a small row/time
budget, and let the actual device report guide neural batch size. Each run writes to a new
candidate directory.



In [ ]:
ASSET_PROVENANCE = (
    ""  # exact asset/version/license; required when enabling local pretrained features
)
competition = CompetitionConfig(
    data_dir=DATA_DIR,
    output_dir=OUTPUT,
    seed=SEED,
    num_workers=0,
    pretrained_allowed=PRETRAINED_ALLOWED,
    time_budget_minutes=TIME_BUDGET_MINUTES,
)
competition.prepare()
budget = TimeBudget(TIME_BUDGET_MINUTES, reserve_minutes=DELIVERY_RESERVE_MINUTES)
print(gpu_report())
print("Local data:", DATA_DIR, "New candidate:", OUTPUT)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>ASSET_PROVENANCE = (
    &quot;&quot;  # exact asset/version/license; required when enabling local pretrained features
)
competition = CompetitionConfig(
    data_dir=DATA_DIR,
    output_dir=OUTPUT,
    seed=SEED,
    num_workers=0,
    pretrained_allowed=PRETRAINED_ALLOWED,
    time_budget_minutes=TIME_BUDGET_MINUTES,
)
competition.prepare()
budget = TimeBudget(TIME_BUDGET_MINUTES, reserve_minutes=DELIVERY_RESERVE_MINUTES)
print(gpu_report())
print(&quot;Local data:&quot;, DATA_DIR, &quot;New candidate:&quot;, OUTPUT)

</code></pre>
</details>


## 1a. Optional dataset acquisition

The cell is skipped by default. For online preparation, first accept this competition's
rules on its Kaggle page and configure credentials using the
[official authentication
guide](https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication).
Set DOWNLOAD_DATA=True below. Local uv sync includes the official Kaggle client. In a copied
notebook/Colab runtime without it, install only `kaggle>=2.2,<3` before running this cell;
do not install this project or replace Colab's torch/NumPy/CUDA stack.

Use KAGGLE_API_TOKEN in your environment, the client's configured token/legacy credential
file, or enable PROMPT_FOR_KAGGLE_TOKEN for hidden input. Never paste a token into saved code.
No rules are accepted automatically, no submission is uploaded, and no weights are fetched.
Downloads can be several GB: allow space for the cached ZIP plus extracted data. DATA_DIR
must be a dedicated empty folder (or a completed download from this cell). Existing manual
data/fixtures are left untouched; keep DOWNLOAD_DATA=False to use them.

This cell reuses a completed dataset/cache, expands nested train/test/CSV archives, and
checks the expected layout. On Kaggle itself, attach competition data in the notebook UI,
point DATA_DIR at its mounted folder, and leave this switch off. For restricted-room drills,
prepare data beforehand and keep this cell disabled. The training timer restarts afterward.



### Supplied helper — `_json_default`

Run this supplied definition before its callers. Its inputs and invariants are explained in this
section.




In [ ]:
def _json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if is_dataclass(value):
        return asdict(value)
    if hasattr(value, "tolist"):
        return value.tolist()
    raise TypeError(f"Cannot serialize {type(value).__name__} as JSON")




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def _json_default(value: Any) -&gt; Any:
    if isinstance(value, Path):
        return str(value)
    if is_dataclass(value):
        return asdict(value)
    if hasattr(value, &quot;tolist&quot;):
        return value.tolist()
    raise TypeError(f&quot;Cannot serialize {type(value).__name__} as JSON&quot;)


</code></pre>
</details>


### Supplied helper — `_write`

Run this supplied definition before its callers. Its inputs and invariants are explained in this
section.




In [ ]:
def _write(value: Any, path: Path | str, kind: str) -> Path:
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f".{destination.name}.{uuid4().hex}.tmp")
    try:
        if kind == "json":
            temporary.write_text(
                json.dumps(
                    value, default=_json_default, indent=2, ensure_ascii=False, allow_nan=False
                ),
                encoding="utf-8",
            )
        elif kind == "pickle":
            with temporary.open("wb") as handle:
                pickle.dump(value, handle, protocol=pickle.HIGHEST_PROTOCOL)
        else:
            import torch

            torch.save(value, temporary)
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def _write(value: Any, path: Path | str, kind: str) -&gt; Path:
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f&quot;.{destination.name}.{uuid4().hex}.tmp&quot;)
    try:
        if kind == &quot;json&quot;:
            temporary.write_text(
                json.dumps(
                    value, default=_json_default, indent=2, ensure_ascii=False, allow_nan=False
                ),
                encoding=&quot;utf-8&quot;,
            )
        elif kind == &quot;pickle&quot;:
            with temporary.open(&quot;wb&quot;) as handle:
                pickle.dump(value, handle, protocol=pickle.HIGHEST_PROTOCOL)
        else:
            import torch

            torch.save(value, temporary)
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination


</code></pre>
</details>


### Supplied helper — `save_json`

Atomically save UTF-8 metadata, including paths, dataclasses, and NumPy values.




In [ ]:
def save_json(value: Any, path: Path | str) -> Path:
    """Atomically save UTF-8 metadata, including paths, dataclasses, and NumPy values."""
    return _write(value, path, "json")




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def save_json(value: Any, path: Path | str) -&gt; Path:
    &quot;&quot;&quot;Atomically save UTF-8 metadata, including paths, dataclasses, and NumPy values.&quot;&quot;&quot;
    return _write(value, path, &quot;json&quot;)


</code></pre>
</details>


In [ ]:
import os
import subprocess
from getpass import getpass
from importlib.util import find_spec

DOWNLOAD_DATA = False
KAGGLE_COMPETITION = "global-wheat-detection"
EXPECTED_DATA_PATHS = ("train.csv", "sample_submission.csv", "train", "test")
DOWNLOAD_CACHE = DATA_DIR.parent / "_downloads" / KAGGLE_COMPETITION
DOWNLOAD_TIMEOUT_SECONDS = 7200
PROMPT_FOR_KAGGLE_TOKEN = "google.colab" in sys.modules



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>import os
import subprocess
from getpass import getpass
from importlib.util import find_spec

DOWNLOAD_DATA = False
KAGGLE_COMPETITION = &quot;global-wheat-detection&quot;
EXPECTED_DATA_PATHS = (&quot;train.csv&quot;, &quot;sample_submission.csv&quot;, &quot;train&quot;, &quot;test&quot;)
DOWNLOAD_CACHE = DATA_DIR.parent / &quot;_downloads&quot; / KAGGLE_COMPETITION
DOWNLOAD_TIMEOUT_SECONDS = 7200
PROMPT_FOR_KAGGLE_TOKEN = &quot;google.colab&quot; in sys.modules

</code></pre>
</details>


### Supplied helper — `extract_dataset_zip`

Extract data without allowing absolute paths, traversal or silent overwrites.




In [ ]:
def extract_dataset_zip(source: Path, destination: Path) -> None:
    """Extract data without allowing absolute paths, traversal or silent overwrites."""
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source) as archive:
        members = archive.infolist()
        if sum(member.file_size for member in members) > shutil.disk_usage(destination).free:
            raise OSError("Insufficient disk space for the next archive")
        for member in members:
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError("Archive contains a path outside the dataset folder")
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            # Duplicated paths indicate a wrong archive/layout; preserve the first file.
            with archive.open(member) as incoming, target.open("xb") as outgoing:
                shutil.copyfileobj(incoming, outgoing)




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def extract_dataset_zip(source: Path, destination: Path) -&gt; None:
    &quot;&quot;&quot;Extract data without allowing absolute paths, traversal or silent overwrites.&quot;&quot;&quot;
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source) as archive:
        members = archive.infolist()
        if sum(member.file_size for member in members) &gt; shutil.disk_usage(destination).free:
            raise OSError(&quot;Insufficient disk space for the next archive&quot;)
        for member in members:
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(&quot;Archive contains a path outside the dataset folder&quot;)
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            # Duplicated paths indicate a wrong archive/layout; preserve the first file.
            with archive.open(member) as incoming, target.open(&quot;xb&quot;) as outgoing:
                shutil.copyfileobj(incoming, outgoing)


</code></pre>
</details>


### Supplied helper — `download_practice_data`

Use the official CLI, then publish the complete extracted folder by renaming it.




In [ ]:
def download_practice_data(
    competition: str,
    destination: Path,
    cache: Path,
    expected_paths: Sequence[str],
    *,
    prompt_token: bool = False,
    timeout_seconds: int = 7200,
) -> Path:
    """Use the official CLI, then publish the complete extracted folder by renaming it."""
    if not competition or any(
        char not in "abcdefghijklmnopqrstuvwxyz0123456789-" for char in competition
    ):
        raise ValueError("Expected a Kaggle competition slug, not a URL or shell command")
    destination, cache = Path(destination).resolve(), Path(cache).resolve()
    if (
        destination == cache
        or cache.is_relative_to(destination)
        or destination.is_relative_to(cache)
    ):
        raise ValueError("Dataset and download cache must be separate folders")
    marker = destination / ".kaggle-source.json"
    if marker.is_file():
        source = json.loads(marker.read_text(encoding="utf-8"))
        if source.get("competition") == competition and all(
            (destination / path).exists() for path in expected_paths
        ):
            print("Reusing completed dataset:", destination)
            return destination
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError(
            "DATA_DIR is not empty/completed; choose a new folder or keep DOWNLOAD_DATA=False"
        )
    cache.mkdir(parents=True, exist_ok=True)
    bundle = cache / f"{competition}.zip"
    if not bundle.is_file() or not zipfile.is_zipfile(bundle):
        if find_spec("kaggle") is None:
            raise ModuleNotFoundError(
                "Install kaggle>=2.2,<3 for online preparation; "
                "local users can run uv sync --locked"
            )
        if prompt_token and not os.environ.get("KAGGLE_API_TOKEN"):
            token = getpass("Kaggle API token (blank uses configured credentials): ").strip()
            if token:
                os.environ["KAGGLE_API_TOKEN"] = token
        # No shell, embedded credentials, auto-acceptance, uploads or package installation.
        command = [
            sys.executable,
            "-m",
            "kaggle",
            "competitions",
            "download",
            competition,
            "--path",
            str(cache),
            "--quiet",
        ]
        if bundle.exists():
            command.append("--force")  # A partial/invalid archive must be fetched again.
        result = subprocess.run(
            command,
            stdin=subprocess.DEVNULL,
            capture_output=True,
            timeout=timeout_seconds,
            check=False,
        )
        if result.returncode:
            # Do not put client output, possible signed URLs or credentials in saved notebooks.
            raise RuntimeError(
                f"Kaggle download failed (exit {result.returncode}). "
                "Check credentials, competition access/rules, internet and disk space. "
                "No dataset was published."
            )
        if not bundle.is_file() or not zipfile.is_zipfile(bundle):
            raise ValueError("Kaggle did not produce a complete competition ZIP; check the cache")
    destination.parent.mkdir(parents=True, exist_ok=True)
    staging = destination.with_name(f".{destination.name}-incoming-{uuid4().hex[:8]}")
    staging.mkdir()
    try:
        extract_dataset_zip(bundle, staging)
        # TGS has train.zip with images/ and masks/; some tasks also wrap CSVs in ZIPs.
        for nested in sorted(staging.rglob("*.zip")):
            folder = nested.parent
            if nested.stem in {"train", "test", "train_images", "test_images"}:
                with zipfile.ZipFile(nested) as archive:
                    names = [item.filename for item in archive.infolist() if not item.is_dir()]
                if names and not all(name.startswith(nested.stem + "/") for name in names):
                    folder = nested.parent / nested.stem
            extract_dataset_zip(nested, folder)
            nested.unlink()  # The outer cached bundle remains available for recovery.
        missing = [path for path in expected_paths if not (staging / path).exists()]
        if missing:
            raise FileNotFoundError(f"Unexpected dataset layout; missing: {missing}")
        save_json(
            {
                "competition": competition,
                "source": f"https://www.kaggle.com/competitions/{competition}/data",
                "expected_paths": list(expected_paths),
            },
            staging / marker.name,
        )
        if destination.exists():
            destination.rmdir()  # Only the previously checked empty directory.
        staging.replace(destination)
    except Exception as error:
        raise RuntimeError(
            "Dataset preparation failed; DATA_DIR was not replaced. "
            f"Inspect partial extraction at {staging}"
        ) from error
    print("Prepared dataset:", destination)
    return destination




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def download_practice_data(
    competition: str,
    destination: Path,
    cache: Path,
    expected_paths: Sequence[str],
    *,
    prompt_token: bool = False,
    timeout_seconds: int = 7200,
) -&gt; Path:
    &quot;&quot;&quot;Use the official CLI, then publish the complete extracted folder by renaming it.&quot;&quot;&quot;
    if not competition or any(
        char not in &quot;abcdefghijklmnopqrstuvwxyz0123456789-&quot; for char in competition
    ):
        raise ValueError(&quot;Expected a Kaggle competition slug, not a URL or shell command&quot;)
    destination, cache = Path(destination).resolve(), Path(cache).resolve()
    if (
        destination == cache
        or cache.is_relative_to(destination)
        or destination.is_relative_to(cache)
    ):
        raise ValueError(&quot;Dataset and download cache must be separate folders&quot;)
    marker = destination / &quot;.kaggle-source.json&quot;
    if marker.is_file():
        source = json.loads(marker.read_text(encoding=&quot;utf-8&quot;))
        if source.get(&quot;competition&quot;) == competition and all(
            (destination / path).exists() for path in expected_paths
        ):
            print(&quot;Reusing completed dataset:&quot;, destination)
            return destination
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError(
            &quot;DATA_DIR is not empty/completed; choose a new folder or keep DOWNLOAD_DATA=False&quot;
        )
    cache.mkdir(parents=True, exist_ok=True)
    bundle = cache / f&quot;{competition}.zip&quot;
    if not bundle.is_file() or not zipfile.is_zipfile(bundle):
        if find_spec(&quot;kaggle&quot;) is None:
            raise ModuleNotFoundError(
                &quot;Install kaggle&gt;=2.2,&lt;3 for online preparation; &quot;
                &quot;local users can run uv sync --locked&quot;
            )
        if prompt_token and not os.environ.get(&quot;KAGGLE_API_TOKEN&quot;):
            token = getpass(&quot;Kaggle API token (blank uses configured credentials): &quot;).strip()
            if token:
                os.environ[&quot;KAGGLE_API_TOKEN&quot;] = token
        # No shell, embedded credentials, auto-acceptance, uploads or package installation.
        command = [
            sys.executable,
            &quot;-m&quot;,
            &quot;kaggle&quot;,
            &quot;competitions&quot;,
            &quot;download&quot;,
            competition,
            &quot;--path&quot;,
            str(cache),
            &quot;--quiet&quot;,
        ]
        if bundle.exists():
            command.append(&quot;--force&quot;)  # A partial/invalid archive must be fetched again.
        result = subprocess.run(
            command,
            stdin=subprocess.DEVNULL,
            capture_output=True,
            timeout=timeout_seconds,
            check=False,
        )
        if result.returncode:
            # Do not put client output, possible signed URLs or credentials in saved notebooks.
            raise RuntimeError(
                f&quot;Kaggle download failed (exit {result.returncode}). &quot;
                &quot;Check credentials, competition access/rules, internet and disk space. &quot;
                &quot;No dataset was published.&quot;
            )
        if not bundle.is_file() or not zipfile.is_zipfile(bundle):
            raise ValueError(&quot;Kaggle did not produce a complete competition ZIP; check the cache&quot;)
    destination.parent.mkdir(parents=True, exist_ok=True)
    staging = destination.with_name(f&quot;.{destination.name}-incoming-{uuid4().hex[:8]}&quot;)
    staging.mkdir()
    try:
        extract_dataset_zip(bundle, staging)
        # TGS has train.zip with images/ and masks/; some tasks also wrap CSVs in ZIPs.
        for nested in sorted(staging.rglob(&quot;*.zip&quot;)):
            folder = nested.parent
            if nested.stem in {&quot;train&quot;, &quot;test&quot;, &quot;train_images&quot;, &quot;test_images&quot;}:
                with zipfile.ZipFile(nested) as archive:
                    names = [item.filename for item in archive.infolist() if not item.is_dir()]
                if names and not all(name.startswith(nested.stem + &quot;/&quot;) for name in names):
                    folder = nested.parent / nested.stem
            extract_dataset_zip(nested, folder)
            nested.unlink()  # The outer cached bundle remains available for recovery.
        missing = [path for path in expected_paths if not (staging / path).exists()]
        if missing:
            raise FileNotFoundError(f&quot;Unexpected dataset layout; missing: {missing}&quot;)
        save_json(
            {
                &quot;competition&quot;: competition,
                &quot;source&quot;: f&quot;https://www.kaggle.com/competitions/{competition}/data&quot;,
                &quot;expected_paths&quot;: list(expected_paths),
            },
            staging / marker.name,
        )
        if destination.exists():
            destination.rmdir()  # Only the previously checked empty directory.
        staging.replace(destination)
    except Exception as error:
        raise RuntimeError(
            &quot;Dataset preparation failed; DATA_DIR was not replaced. &quot;
            f&quot;Inspect partial extraction at {staging}&quot;
        ) from error
    print(&quot;Prepared dataset:&quot;, destination)
    return destination


</code></pre>
</details>


In [ ]:
if DOWNLOAD_DATA:
    if DATA_ARCHIVE is not None:
        raise ValueError("Choose either DATA_ARCHIVE staging or DOWNLOAD_DATA, then rerun setup")
    DATA_DIR = download_practice_data(
        KAGGLE_COMPETITION,
        DATA_DIR,
        DOWNLOAD_CACHE,
        EXPECTED_DATA_PATHS,
        prompt_token=PROMPT_FOR_KAGGLE_TOKEN,
        timeout_seconds=DOWNLOAD_TIMEOUT_SECONDS,
    )
source_marker = DATA_DIR / ".kaggle-source.json"
if source_marker.is_file():
    save_json(json.loads(source_marker.read_text(encoding="utf-8")), OUTPUT / "data_source.json")
print("Download enabled:", DOWNLOAD_DATA, "Dataset folder:", DATA_DIR)
budget = TimeBudget(TIME_BUDGET_MINUTES, reserve_minutes=DELIVERY_RESERVE_MINUTES)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>if DOWNLOAD_DATA:
    if DATA_ARCHIVE is not None:
        raise ValueError(&quot;Choose either DATA_ARCHIVE staging or DOWNLOAD_DATA, then rerun setup&quot;)
    DATA_DIR = download_practice_data(
        KAGGLE_COMPETITION,
        DATA_DIR,
        DOWNLOAD_CACHE,
        EXPECTED_DATA_PATHS,
        prompt_token=PROMPT_FOR_KAGGLE_TOKEN,
        timeout_seconds=DOWNLOAD_TIMEOUT_SECONDS,
    )
source_marker = DATA_DIR / &quot;.kaggle-source.json&quot;
if source_marker.is_file():
    save_json(json.loads(source_marker.read_text(encoding=&quot;utf-8&quot;)), OUTPUT / &quot;data_source.json&quot;)
print(&quot;Download enabled:&quot;, DOWNLOAD_DATA, &quot;Dataset folder:&quot;, DATA_DIR)
budget = TimeBudget(TIME_BUDGET_MINUTES, reserve_minutes=DELIVERY_RESERVE_MINUTES)

</code></pre>
</details>


## 2. Understand and load the data contract

One image may contain many boxes, so annotation rows must first be grouped by image_id. The
output is a prediction string of confidence and xywh boxes per image, including a valid empty
string when no box survives.

Read the first rows, column names and actual shapes below. Expected: unique nonempty prediction
IDs; matching test/sample ID sets; a target schema that agrees with the official output. Dataset
size depends on your local files and MAX_ROWS.



In [ ]:
import ast

ID_COLUMN, TARGET_COLUMNS = "image_id", ["PredictionString"]
annotations = pd.read_csv(DATA_DIR / "train.csv", converters={ID_COLUMN: str})
sample = pd.read_csv(DATA_DIR / "sample_submission.csv", converters={ID_COLUMN: str})
parsed = np.asarray([ast.literal_eval(text) for text in annotations["bbox"]], dtype=np.float32)
assert parsed.shape == (len(annotations), 4)
assert np.isfinite(parsed).all() and (parsed[:, 2:] > 0).all()
annotations[["xmin", "ymin"]] = parsed[:, :2]
annotations[["xmax", "ymax"]] = parsed[:, :2] + parsed[:, 2:]
annotations["label"] = 1  # background is zero in torchvision
annotations["image"] = "train/" + annotations[ID_COLUMN] + IMAGE_SUFFIX
train = (
    annotations.groupby(ID_COLUMN, sort=False)
    .agg(
        source=("source", "first"),
        box_count=("bbox", "size"),
        image=("image", "first"),
        width=("width", "first"),
        height=("height", "first"),
    )
    .reset_index()
)
assert annotations.groupby(ID_COLUMN)["source"].nunique().max() == 1
test = sample[[ID_COLUMN]].copy()



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>import ast

ID_COLUMN, TARGET_COLUMNS = &quot;image_id&quot;, [&quot;PredictionString&quot;]
annotations = pd.read_csv(DATA_DIR / &quot;train.csv&quot;, converters={ID_COLUMN: str})
sample = pd.read_csv(DATA_DIR / &quot;sample_submission.csv&quot;, converters={ID_COLUMN: str})
parsed = np.asarray([ast.literal_eval(text) for text in annotations[&quot;bbox&quot;]], dtype=np.float32)
assert parsed.shape == (len(annotations), 4)
assert np.isfinite(parsed).all() and (parsed[:, 2:] &gt; 0).all()
annotations[[&quot;xmin&quot;, &quot;ymin&quot;]] = parsed[:, :2]
annotations[[&quot;xmax&quot;, &quot;ymax&quot;]] = parsed[:, :2] + parsed[:, 2:]
annotations[&quot;label&quot;] = 1  # background is zero in torchvision
annotations[&quot;image&quot;] = &quot;train/&quot; + annotations[ID_COLUMN] + IMAGE_SUFFIX
train = (
    annotations.groupby(ID_COLUMN, sort=False)
    .agg(
        source=(&quot;source&quot;, &quot;first&quot;),
        box_count=(&quot;bbox&quot;, &quot;size&quot;),
        image=(&quot;image&quot;, &quot;first&quot;),
        width=(&quot;width&quot;, &quot;first&quot;),
        height=(&quot;height&quot;, &quot;first&quot;),
    )
    .reset_index()
)
assert annotations.groupby(ID_COLUMN)[&quot;source&quot;].nunique().max() == 1
test = sample[[ID_COLUMN]].copy()

</code></pre>
</details>


In [ ]:
test["image"] = "test/" + test[ID_COLUMN] + IMAGE_SUFFIX
if MAX_ROWS is not None and len(train) > MAX_ROWS:
    train = train.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)
for name, frame in (("train", train), ("test", test), ("sample", sample)):
    if (
        frame[ID_COLUMN].isna().any()
        or frame[ID_COLUMN].eq("").any()
        or frame[ID_COLUMN].duplicated().any()
    ):
        raise ValueError(f"{name} must have unique, nonempty prediction-unit IDs")
assert not set(train[ID_COLUMN]) & set(test[ID_COLUMN])
assert set(test[ID_COLUMN]) == set(sample[ID_COLUMN])
assert sample.columns.tolist() == [ID_COLUMN, *TARGET_COLUMNS]
print(train.head(3).to_string(index=False))
print("Shapes:", train.shape, test.shape, sample.shape)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>test[&quot;image&quot;] = &quot;test/&quot; + test[ID_COLUMN] + IMAGE_SUFFIX
if MAX_ROWS is not None and len(train) &gt; MAX_ROWS:
    train = train.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)
for name, frame in ((&quot;train&quot;, train), (&quot;test&quot;, test), (&quot;sample&quot;, sample)):
    if (
        frame[ID_COLUMN].isna().any()
        or frame[ID_COLUMN].eq(&quot;&quot;).any()
        or frame[ID_COLUMN].duplicated().any()
    ):
        raise ValueError(f&quot;{name} must have unique, nonempty prediction-unit IDs&quot;)
assert not set(train[ID_COLUMN]) &amp; set(test[ID_COLUMN])
assert set(test[ID_COLUMN]) == set(sample[ID_COLUMN])
assert sample.columns.tolist() == [ID_COLUMN, *TARGET_COLUMNS]
print(train.head(3).to_string(index=False))
print(&quot;Shapes:&quot;, train.shape, test.shape, sample.shape)

</code></pre>
</details>


In [ ]:
assert len(train) > 0 and len(test) == len(sample)
assert train[ID_COLUMN].is_unique and sample[ID_COLUMN].is_unique
print("Data-contract checks passed: prediction units and sample coverage.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>assert len(train) &gt; 0 and len(test) == len(sample)
assert train[ID_COLUMN].is_unique and sample[ID_COLUMN].is_unique
print(&quot;Data-contract checks passed: prediction units and sample coverage.&quot;)

</code></pre>
</details>


### Stop and explain

What is the prediction unit, and which columns may enter the model?

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

One image may contain many boxes, so annotation rows must first be grouped by image_id. The
output is a prediction string of confidence and xywh boxes per image, including a valid empty
string when no box survives.

</details>



## 3. Inspect examples and distributions

Inspect the plotted examples before selecting a model. Write observations before reading the
next section.

How do box counts and sizes vary by acquisition source? Do drawn boxes fit the actual image? Are
there negative images absent from the annotation table, and are bbox values xywh or xyxy?



In [ ]:
from matplotlib.patches import Rectangle

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (_, row) in zip(axes, train.head(3).iterrows(), strict=False):
    with Image.open(DATA_DIR / row["image"]) as image:
        ax.imshow(image.convert("RGB"))
    boxes = annotations.loc[annotations[ID_COLUMN] == row[ID_COLUMN]]
    for _, box in boxes.iterrows():
        ax.add_patch(
            Rectangle(
                (box["xmin"], box["ymin"]),
                box["xmax"] - box["xmin"],
                box["ymax"] - box["ymin"],
                fill=False,
                edgecolor="red",
            )
        )
    ax.set_title(f"{row['box_count']} boxes; {row['source']}")
    ax.axis("off")
fig.savefig(OUTPUT / "inspection.png")
plt.show()
plt.close(fig)
print("Per-source images:", train["source"].value_counts().to_dict())



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>from matplotlib.patches import Rectangle

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (_, row) in zip(axes, train.head(3).iterrows(), strict=False):
    with Image.open(DATA_DIR / row[&quot;image&quot;]) as image:
        ax.imshow(image.convert(&quot;RGB&quot;))
    boxes = annotations.loc[annotations[ID_COLUMN] == row[ID_COLUMN]]
    for _, box in boxes.iterrows():
        ax.add_patch(
            Rectangle(
                (box[&quot;xmin&quot;], box[&quot;ymin&quot;]),
                box[&quot;xmax&quot;] - box[&quot;xmin&quot;],
                box[&quot;ymax&quot;] - box[&quot;ymin&quot;],
                fill=False,
                edgecolor=&quot;red&quot;,
            )
        )
    ax.set_title(f&quot;{row[&#x27;box_count&#x27;]} boxes; {row[&#x27;source&#x27;]}&quot;)
    ax.axis(&quot;off&quot;)
fig.savefig(OUTPUT / &quot;inspection.png&quot;)
plt.show()
plt.close(fig)
print(&quot;Per-source images:&quot;, train[&quot;source&quot;].value_counts().to_dict())

</code></pre>
</details>


In [ ]:
print(
    "Box-size quantiles:",
    pd.DataFrame(parsed[:, 2:], columns=["width", "height"]).quantile([0, 0.5, 0.9, 1]),
)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>print(
    &quot;Box-size quantiles:&quot;,
    pd.DataFrame(parsed[:, 2:], columns=[&quot;width&quot;, &quot;height&quot;]).quantile([0, 0.5, 0.9, 1]),
)

</code></pre>
</details>


### Stop and explain

Record three observations: one distribution issue, one suspicious example, and one possible
leakage source. What experiment does each justify?

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

Use the actual plots and rows; there is no fixed answer independent of your pack. Convert each
observation into a falsifiable check before spending training time.

</details>



## 4. Build and test the validation split

Split image records, never annotation rows. Keep acquisition sources together when available. If
one image's boxes appear on both sides, the validation score no longer measures generalization
to unseen images.



In [ ]:
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, train_test_split



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, train_test_split

</code></pre>
</details>


### Supplied helper — `SplitResult`

Training/validation tables plus their original positional indexes.




In [ ]:
@dataclass(slots=True)
class SplitResult:
    """Training/validation tables plus their original positional indexes."""

    train: pd.DataFrame
    valid: pd.DataFrame
    train_indices: np.ndarray
    valid_indices: np.ndarray




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>@dataclass(slots=True)
class SplitResult:
    &quot;&quot;&quot;Training/validation tables plus their original positional indexes.&quot;&quot;&quot;

    train: pd.DataFrame
    valid: pd.DataFrame
    train_indices: np.ndarray
    valid_indices: np.ndarray


</code></pre>
</details>


### Supplied helper — `_stratification_key`

Run this supplied definition before its callers. Its inputs and invariants are explained in this
section.




In [ ]:
def _stratification_key(frame: pd.DataFrame, targets: Sequence[str]) -> pd.Series:
    if not targets:
        raise ValueError("At least one target is required for stratification")
    if len(targets) == 1:
        return frame[targets[0]].astype(str)
    return frame[list(targets)].astype(str).agg("|".join, axis=1)




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def _stratification_key(frame: pd.DataFrame, targets: Sequence[str]) -&gt; pd.Series:
    if not targets:
        raise ValueError(&quot;At least one target is required for stratification&quot;)
    if len(targets) == 1:
        return frame[targets[0]].astype(str)
    return frame[list(targets)].astype(str).agg(&quot;|&quot;.join, axis=1)


</code></pre>
</details>


### Supplied helper — `make_split`

Create a deterministic holdout with leakage-aware group and time options.




In [ ]:
def make_split(
    frame: pd.DataFrame,
    *,
    target_columns: str | Sequence[str] | None = None,
    group_column: str | None = None,
    time_column: str | None = None,
    valid_size: float = 0.2,
    seed: int = 42,
) -> SplitResult:
    """Create a deterministic holdout with leakage-aware group and time options."""
    if not 0 < valid_size < 1:
        raise ValueError("valid_size must be between 0 and 1")
    if len(frame) < 2:
        raise ValueError("At least two rows are required")
    targets = [target_columns] if isinstance(target_columns, str) else list(target_columns or [])
    for column in [*targets, group_column, time_column]:
        if column is not None and column not in frame.columns:
            raise KeyError(f"Missing split column: {column}")

    indices = np.arange(len(frame))
    if time_column:
        timestamps = pd.to_datetime(frame[time_column], errors="raise")
        if timestamps.isna().any():
            raise ValueError("Time split requires complete timestamps")
        order = np.argsort(timestamps.to_numpy(), kind="stable")
        ordered_times = timestamps.to_numpy()[order]
        boundaries = np.flatnonzero(ordered_times[1:] != ordered_times[:-1]) + 1
        if not len(boundaries):
            raise ValueError("Need at least two distinct times for a strict chronological split")
        desired = len(frame) * (1 - valid_size)
        cut = int(boundaries[np.argmin(abs(boundaries - desired))])
        train_idx, valid_idx = order[:cut], order[cut:]
    elif group_column and targets:
        folds = max(2, round(1 / valid_size))
        splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
        try:
            train_idx, valid_idx = next(
                splitter.split(indices, _stratification_key(frame, targets), frame[group_column])
            )
        except ValueError:
            fallback = GroupShuffleSplit(n_splits=1, test_size=valid_size, random_state=seed)
            train_idx, valid_idx = next(fallback.split(indices, groups=frame[group_column]))
    elif group_column:
        splitter = GroupShuffleSplit(n_splits=1, test_size=valid_size, random_state=seed)
        train_idx, valid_idx = next(splitter.split(indices, groups=frame[group_column]))
    else:
        stratify = _stratification_key(frame, targets) if targets else None
        try:
            train_idx, valid_idx = train_test_split(
                indices, test_size=valid_size, random_state=seed, stratify=stratify
            )
        except ValueError:
            train_idx, valid_idx = train_test_split(
                indices, test_size=valid_size, random_state=seed, stratify=None
            )

    if group_column:
        overlap = set(frame.iloc[train_idx][group_column]) & set(
            frame.iloc[valid_idx][group_column]
        )
        if overlap:
            raise RuntimeError(f"Group leakage detected for {len(overlap)} groups")
    return SplitResult(
        train=frame.iloc[train_idx].reset_index(drop=True),
        valid=frame.iloc[valid_idx].reset_index(drop=True),
        train_indices=np.asarray(train_idx),
        valid_indices=np.asarray(valid_idx),
    )




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def make_split(
    frame: pd.DataFrame,
    *,
    target_columns: str | Sequence[str] | None = None,
    group_column: str | None = None,
    time_column: str | None = None,
    valid_size: float = 0.2,
    seed: int = 42,
) -&gt; SplitResult:
    &quot;&quot;&quot;Create a deterministic holdout with leakage-aware group and time options.&quot;&quot;&quot;
    if not 0 &lt; valid_size &lt; 1:
        raise ValueError(&quot;valid_size must be between 0 and 1&quot;)
    if len(frame) &lt; 2:
        raise ValueError(&quot;At least two rows are required&quot;)
    targets = [target_columns] if isinstance(target_columns, str) else list(target_columns or [])
    for column in [*targets, group_column, time_column]:
        if column is not None and column not in frame.columns:
            raise KeyError(f&quot;Missing split column: {column}&quot;)

    indices = np.arange(len(frame))
    if time_column:
        timestamps = pd.to_datetime(frame[time_column], errors=&quot;raise&quot;)
        if timestamps.isna().any():
            raise ValueError(&quot;Time split requires complete timestamps&quot;)
        order = np.argsort(timestamps.to_numpy(), kind=&quot;stable&quot;)
        ordered_times = timestamps.to_numpy()[order]
        boundaries = np.flatnonzero(ordered_times[1:] != ordered_times[:-1]) + 1
        if not len(boundaries):
            raise ValueError(&quot;Need at least two distinct times for a strict chronological split&quot;)
        desired = len(frame) * (1 - valid_size)
        cut = int(boundaries[np.argmin(abs(boundaries - desired))])
        train_idx, valid_idx = order[:cut], order[cut:]
    elif group_column and targets:
        folds = max(2, round(1 / valid_size))
        splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
        try:
            train_idx, valid_idx = next(
                splitter.split(indices, _stratification_key(frame, targets), frame[group_column])
            )
        except ValueError:
            fallback = GroupShuffleSplit(n_splits=1, test_size=valid_size, random_state=seed)
            train_idx, valid_idx = next(fallback.split(indices, groups=frame[group_column]))
    elif group_column:
        splitter = GroupShuffleSplit(n_splits=1, test_size=valid_size, random_state=seed)
        train_idx, valid_idx = next(splitter.split(indices, groups=frame[group_column]))
    else:
        stratify = _stratification_key(frame, targets) if targets else None
        try:
            train_idx, valid_idx = train_test_split(
                indices, test_size=valid_size, random_state=seed, stratify=stratify
            )
        except ValueError:
            train_idx, valid_idx = train_test_split(
                indices, test_size=valid_size, random_state=seed, stratify=None
            )

    if group_column:
        overlap = set(frame.iloc[train_idx][group_column]) &amp; set(
            frame.iloc[valid_idx][group_column]
        )
        if overlap:
            raise RuntimeError(f&quot;Group leakage detected for {len(overlap)} groups&quot;)
    return SplitResult(
        train=frame.iloc[train_idx].reset_index(drop=True),
        valid=frame.iloc[valid_idx].reset_index(drop=True),
        train_indices=np.asarray(train_idx),
        valid_indices=np.asarray(valid_idx),
    )


</code></pre>
</details>


### Exercise 1 — `create_holdout`

Input: a row-per-prediction table and its actual leakage columns. Return: the split record and
independent training/validation copies. The union must retain all rows; identities must not
cross folds.

<details>
<summary>Hint 1 — choose the operation</summary>

Split image records, never annotation rows. Keep acquisition sources together when available. If
one image's boxes appear on both sides, the validation score no longer measures generalization
to unseen images.

</details>

<details>
<summary>Hint 2 — implementation direction</summary>

Use make_split with valid_size=0.2; return copies of its train and valid tables.

</details>




In [ ]:
def create_holdout(
    frame: pd.DataFrame,
    target_columns: Any = None,
    group_column: str | None = None,
    time_column: str | None = None,
    seed: int = 42,
) -> Any:
    """Input: a row-per-prediction table and its actual leakage columns. Return: the split
    record and independent training/validation copies. The union must retain all rows;
    identities must not cross folds.
    """
    raise NotImplementedError('TODO: implement create_holdout using the stated contract')
    return split, tr, va




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def create_holdout(
    frame: pd.DataFrame,
    target_columns: Any = None,
    group_column: str | None = None,
    time_column: str | None = None,
    seed: int = 42,
) -&gt; Any:
    &quot;&quot;&quot;Input: a row-per-prediction table and its actual leakage columns. Return: the split
    record and independent training/validation copies. The union must retain all rows;
    identities must not cross folds.
    &quot;&quot;&quot;
    split = make_split(
        frame,
        target_columns=target_columns,
        group_column=group_column,
        time_column=time_column,
        valid_size=0.2,
        seed=seed,
    )
    tr, va = split.train.copy(), split.valid.copy()
    return split, tr, va


</code></pre>
</details>


**Check before continuing.** Predict the result, run the small test below, and use its assertion
message to locate an error. Expected: all assertions pass and the printed success message
appears. These checks do not fit the real candidate.



In [ ]:
toy_frame = pd.DataFrame({"id": range(12), "group": np.repeat(range(6), 2)})
toy_split, toy_train, toy_valid = create_holdout(toy_frame, group_column="group", seed=7)
assert len(toy_train) + len(toy_valid) == 12, "The split must retain every row"
assert not set(toy_train["group"]) & set(toy_valid["group"]), "Repeated identities leaked"
assert not set(toy_split.train_indices) & set(toy_split.valid_indices)
toy_again = create_holdout(toy_frame, group_column="group", seed=7)[0]
np.testing.assert_array_equal(toy_split.valid_indices, toy_again.valid_indices)
print("Split checks passed: complete, group-disjoint and reproducible.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>toy_frame = pd.DataFrame({&quot;id&quot;: range(12), &quot;group&quot;: np.repeat(range(6), 2)})
toy_split, toy_train, toy_valid = create_holdout(toy_frame, group_column=&quot;group&quot;, seed=7)
assert len(toy_train) + len(toy_valid) == 12, &quot;The split must retain every row&quot;
assert not set(toy_train[&quot;group&quot;]) &amp; set(toy_valid[&quot;group&quot;]), &quot;Repeated identities leaked&quot;
assert not set(toy_split.train_indices) &amp; set(toy_split.valid_indices)
toy_again = create_holdout(toy_frame, group_column=&quot;group&quot;, seed=7)[0]
np.testing.assert_array_equal(toy_split.valid_indices, toy_again.valid_indices)
print(&quot;Split checks passed: complete, group-disjoint and reproducible.&quot;)

</code></pre>
</details>


In [ ]:
split, tr, va = create_holdout(
    train, target_columns=None, group_column=GROUP_COLUMN, time_column=TIME_COLUMN, seed=SEED
)
assert not set(tr[ID_COLUMN]) & set(va[ID_COLUMN])
if GROUP_COLUMN:
    assert not set(tr[GROUP_COLUMN]) & set(va[GROUP_COLUMN])
if TIME_COLUMN:
    assert pd.to_datetime(tr[TIME_COLUMN]).max() < pd.to_datetime(va[TIME_COLUMN]).min()
print("Image counts:", len(tr), len(va), "Validation sources:", va["source"].unique())



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>split, tr, va = create_holdout(
    train, target_columns=None, group_column=GROUP_COLUMN, time_column=TIME_COLUMN, seed=SEED
)
assert not set(tr[ID_COLUMN]) &amp; set(va[ID_COLUMN])
if GROUP_COLUMN:
    assert not set(tr[GROUP_COLUMN]) &amp; set(va[GROUP_COLUMN])
if TIME_COLUMN:
    assert pd.to_datetime(tr[TIME_COLUMN]).max() &lt; pd.to_datetime(va[TIME_COLUMN]).min()
print(&quot;Image counts:&quot;, len(tr), len(va), &quot;Validation sources:&quot;, va[&quot;source&quot;].unique())

</code></pre>
</details>


In [ ]:
assert len(tr) > 0 and len(va) > 0
assert not set(tr[ID_COLUMN]) & set(va[ID_COLUMN])
print("Actual-fold checks passed: nonempty disjoint prediction IDs.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>assert len(tr) &gt; 0 and len(va) &gt; 0
assert not set(tr[ID_COLUMN]) &amp; set(va[ID_COLUMN])
print(&quot;Actual-fold checks passed: nonempty disjoint prediction IDs.&quot;)

</code></pre>
</details>


### Stop and explain

Would a random row split measure the generalization you need here? Identify the boundary that
would leak information.

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

Split image records, never annotation rows. Keep acquisition sources together when available. If
one image's boxes appear on both sides, the validation score no longer measures generalization
to unseen images.

</details>



## 5. Measure imbalance and choose a response

There is one foreground class, but image density and source frequencies can be uneven. Derive
optional density sampling weights from training images only. Do not treat every box as an
independent training example.

Per-image score uses $TP/(TP+FP+FN)$ averaged over IoU thresholds. This notebook exposes strict
IoU/confidence boundaries and inclusive-pixel areas; it does not substitute COCO AP.



In [ ]:
from torch.utils.data import WeightedRandomSampler



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>from torch.utils.data import WeightedRandomSampler

</code></pre>
</details>


### Exercise 2 — `density_sampling`

Input: one box-count value per training image. Return: density bins, frequencies and per-image
sampling weights [N]. This is not a box-level split.

<details>
<summary>Hint 1 — choose the operation</summary>

There is one foreground class, but image density and source frequencies can be uneven. Derive
optional density sampling weights from training images only. Do not treat every box as an
independent training example.

</details>

<details>
<summary>Hint 2 — implementation direction</summary>

Bin image density, map each image to its bin frequency, then take the reciprocal.

</details>




In [ ]:
def density_sampling(box_counts: pd.Series) -> Any:
    """Input: one box-count value per training image. Return: density bins, frequencies and
    per-image sampling weights [N]. This is not a box-level split.
    """
    raise NotImplementedError('TODO: implement density_sampling using the stated contract')
    return density_bins, frequency, sampling_weights




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def density_sampling(box_counts: pd.Series) -&gt; Any:
    &quot;&quot;&quot;Input: one box-count value per training image. Return: density bins, frequencies and
    per-image sampling weights [N]. This is not a box-level split.
    &quot;&quot;&quot;
    density_bins = pd.cut(box_counts, bins=[-1, 2, 10, 50, 1e9])
    frequency = density_bins.value_counts()
    sampling_weights = (1 / density_bins.map(frequency).astype(float)).to_numpy()
    return density_bins, frequency, sampling_weights


</code></pre>
</details>


**Check before continuing.** Predict the result, run the small test below, and use its assertion
message to locate an error. Expected: all assertions pass and the printed success message
appears. These checks do not fit the real candidate.



In [ ]:
toy_bins, toy_frequency, toy_sampling = density_sampling(pd.Series([1, 1, 8]))
np.testing.assert_allclose(toy_sampling, [0.5, 0.5, 1.0])
assert np.isfinite(toy_sampling).all()
print("Density checks passed: rare density bins receive larger training sampling weights.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>toy_bins, toy_frequency, toy_sampling = density_sampling(pd.Series([1, 1, 8]))
np.testing.assert_allclose(toy_sampling, [0.5, 0.5, 1.0])
assert np.isfinite(toy_sampling).all()
print(&quot;Density checks passed: rare density bins receive larger training sampling weights.&quot;)

</code></pre>
</details>


In [ ]:
density_bins, frequency, sampling_weights = density_sampling(tr["box_count"])
print("One foreground class; no invented foreground class-balancing weights")
print("Density bins:", frequency.to_dict(), "Sampling enabled:", USE_IMBALANCE_WEIGHTS)
tr["box_count"].hist(bins=20)
plt.title("Training boxes per image")
plt.savefig(OUTPUT / "imbalance.png")
plt.show()
plt.close()



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>density_bins, frequency, sampling_weights = density_sampling(tr[&quot;box_count&quot;])
print(&quot;One foreground class; no invented foreground class-balancing weights&quot;)
print(&quot;Density bins:&quot;, frequency.to_dict(), &quot;Sampling enabled:&quot;, USE_IMBALANCE_WEIGHTS)
tr[&quot;box_count&quot;].hist(bins=20)
plt.title(&quot;Training boxes per image&quot;)
plt.savefig(OUTPUT / &quot;imbalance.png&quot;)
plt.show()
plt.close()

</code></pre>
</details>


### Stop and explain

Does changing training weights also change the official score or validation distribution? What
would justify enabling the response?

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

There is one foreground class, but image density and source frequencies can be uneven. Derive
optional density sampling weights from training images only. Do not treat every box as an
independent training example.

</details>



## 6. Build features or one correct batch

A dataset item returns an image tensor and a variable-length target dictionary. The collate
function returns lists, not stacked box arrays. Foreground labels start at 1 because torchvision
reserves 0 for background.



In [ ]:
from torch.utils.data import Dataset
from torchvision.transforms import functional as TF



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>from torch.utils.data import Dataset
from torchvision.transforms import functional as TF

</code></pre>
</details>


### Worked component — `BoxDetectionDataset`

Read detection annotations with one row per bounding box.

Expected coordinates are absolute ``xmin, ymin, xmax, ymax`` values. Rows sharing an image
path are grouped into one target dictionary in the format expected by torchvision detectors.
Class index 0 is reserved for background, so foreground labels must start at 1.




In [ ]:
class BoxDetectionDataset(Dataset[tuple[torch.Tensor, dict[str, torch.Tensor]]]):
    """Read detection annotations with one row per bounding box.

    Expected coordinates are absolute ``xmin, ymin, xmax, ymax`` values. Rows sharing an image
    path are grouped into one target dictionary in the format expected by torchvision detectors.
    Class index 0 is reserved for background, so foreground labels must start at 1.
    """

    def __init__(
        self,
        frame: pd.DataFrame,
        *,
        image_column: str = "image",
        label_column: str = "label",
        box_columns: tuple[str, str, str, str] = ("xmin", "ymin", "xmax", "ymax"),
        root: Path | str = ".",
    ) -> None:
        """Validate annotation columns and group rows by image path."""
        required = {image_column, label_column, *box_columns}
        missing = required - set(frame.columns)
        if missing:
            raise ValueError(f"Detection table is missing columns: {sorted(missing)}")
        self.groups = list(frame.groupby(image_column, sort=False))
        self.label_column = label_column
        self.box_columns = box_columns
        self.root = Path(root)

    def __len__(self) -> int:
        """Return the number of distinct images."""
        return len(self.groups)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        """Load one RGB image and all of its boxes and labels."""
        relative_path, rows = self.groups[index]
        with Image.open(self.root / str(relative_path)) as source:
            image = TF.pil_to_tensor(source.convert("RGB")).float() / 255
        boxes = torch.as_tensor(rows.loc[:, self.box_columns].to_numpy(), dtype=torch.float32)
        labels = torch.as_tensor(rows[self.label_column].to_numpy(), dtype=torch.int64)
        if (labels < 1).any():
            raise ValueError("Torchvision foreground detection labels must start at 1")
        if not torch.isfinite(boxes).all() or (boxes[:, 2:] <= boxes[:, :2]).any():
            raise ValueError("Boxes must be finite exclusive xyxy rectangles with positive area")
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([index]),
            "area": (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]),
            "iscrowd": torch.zeros(len(boxes), dtype=torch.int64),
        }
        return image, target




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>class BoxDetectionDataset(Dataset[tuple[torch.Tensor, dict[str, torch.Tensor]]]):
    &quot;&quot;&quot;Read detection annotations with one row per bounding box.

    Expected coordinates are absolute ``xmin, ymin, xmax, ymax`` values. Rows sharing an image
    path are grouped into one target dictionary in the format expected by torchvision detectors.
    Class index 0 is reserved for background, so foreground labels must start at 1.
    &quot;&quot;&quot;

    def __init__(
        self,
        frame: pd.DataFrame,
        *,
        image_column: str = &quot;image&quot;,
        label_column: str = &quot;label&quot;,
        box_columns: tuple[str, str, str, str] = (&quot;xmin&quot;, &quot;ymin&quot;, &quot;xmax&quot;, &quot;ymax&quot;),
        root: Path | str = &quot;.&quot;,
    ) -&gt; None:
        &quot;&quot;&quot;Validate annotation columns and group rows by image path.&quot;&quot;&quot;
        required = {image_column, label_column, *box_columns}
        missing = required - set(frame.columns)
        if missing:
            raise ValueError(f&quot;Detection table is missing columns: {sorted(missing)}&quot;)
        self.groups = list(frame.groupby(image_column, sort=False))
        self.label_column = label_column
        self.box_columns = box_columns
        self.root = Path(root)

    def __len__(self) -&gt; int:
        &quot;&quot;&quot;Return the number of distinct images.&quot;&quot;&quot;
        return len(self.groups)

    def __getitem__(self, index: int) -&gt; tuple[torch.Tensor, dict[str, torch.Tensor]]:
        &quot;&quot;&quot;Load one RGB image and all of its boxes and labels.&quot;&quot;&quot;
        relative_path, rows = self.groups[index]
        with Image.open(self.root / str(relative_path)) as source:
            image = TF.pil_to_tensor(source.convert(&quot;RGB&quot;)).float() / 255
        boxes = torch.as_tensor(rows.loc[:, self.box_columns].to_numpy(), dtype=torch.float32)
        labels = torch.as_tensor(rows[self.label_column].to_numpy(), dtype=torch.int64)
        if (labels &lt; 1).any():
            raise ValueError(&quot;Torchvision foreground detection labels must start at 1&quot;)
        if not torch.isfinite(boxes).all() or (boxes[:, 2:] &lt;= boxes[:, :2]).any():
            raise ValueError(&quot;Boxes must be finite exclusive xyxy rectangles with positive area&quot;)
        target = {
            &quot;boxes&quot;: boxes,
            &quot;labels&quot;: labels,
            &quot;image_id&quot;: torch.tensor([index]),
            &quot;area&quot;: (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]),
            &quot;iscrowd&quot;: torch.zeros(len(boxes), dtype=torch.int64),
        }
        return image, target


</code></pre>
</details>


### Worked component — `DetectionInferenceDataset`

Read an ordered image manifest without resizing or normalizing detector inputs.




In [ ]:
class DetectionInferenceDataset(Dataset[Any]):
    """Read an ordered image manifest without resizing or normalizing detector inputs."""

    def __init__(
        self, frame: pd.DataFrame, *, image_column: str = "image", root: Path | str = "."
    ) -> None:
        self.frame = frame.reset_index(drop=True)
        self.image_column = image_column
        self.root = Path(root)

    def __len__(self) -> int:
        """Return the number of inference images."""
        return len(self.frame)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        """Return an RGB tensor and empty target for the variable-size detection collator."""
        with Image.open(self.root / str(self.frame.iloc[index][self.image_column])) as image:
            return TF.pil_to_tensor(image.convert("RGB")).float() / 255, {}




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>class DetectionInferenceDataset(Dataset[Any]):
    &quot;&quot;&quot;Read an ordered image manifest without resizing or normalizing detector inputs.&quot;&quot;&quot;

    def __init__(
        self, frame: pd.DataFrame, *, image_column: str = &quot;image&quot;, root: Path | str = &quot;.&quot;
    ) -&gt; None:
        self.frame = frame.reset_index(drop=True)
        self.image_column = image_column
        self.root = Path(root)

    def __len__(self) -&gt; int:
        &quot;&quot;&quot;Return the number of inference images.&quot;&quot;&quot;
        return len(self.frame)

    def __getitem__(self, index: int) -&gt; tuple[torch.Tensor, dict[str, torch.Tensor]]:
        &quot;&quot;&quot;Return an RGB tensor and empty target for the variable-size detection collator.&quot;&quot;&quot;
        with Image.open(self.root / str(self.frame.iloc[index][self.image_column])) as image:
            return TF.pil_to_tensor(image.convert(&quot;RGB&quot;)).float() / 255, {}


</code></pre>
</details>


### Worked component — `detection_collate`

Collate variable-size images and variable-length detection targets.




In [ ]:
def detection_collate(batch: list[tuple[Any, Any]]) -> tuple[tuple[Any, ...], tuple[Any, ...]]:
    """Collate variable-size images and variable-length detection targets."""
    return tuple(zip(*batch, strict=True))




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def detection_collate(batch: list[tuple[Any, Any]]) -&gt; tuple[tuple[Any, ...], tuple[Any, ...]]:
    &quot;&quot;&quot;Collate variable-size images and variable-length detection targets.&quot;&quot;&quot;
    return tuple(zip(*batch, strict=True))


</code></pre>
</details>


### Worked component — `predict_detections`

Return CPU detection dictionaries in the manifest/dataset order, fitting nothing.




In [ ]:
@torch.inference_mode()
def predict_detections(
    model: nn.Module, loader: Any, *, device: str = "cpu"
) -> list[dict[str, torch.Tensor]]:
    """Return CPU detection dictionaries in the manifest/dataset order, fitting nothing."""
    model.to(device).eval()
    outputs = []
    for images, _ in loader:
        outputs.extend(
            {key: value.cpu() for key, value in prediction.items()}
            for prediction in model([image.to(device) for image in images])
        )
    return outputs




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>@torch.inference_mode()
def predict_detections(
    model: nn.Module, loader: Any, *, device: str = &quot;cpu&quot;
) -&gt; list[dict[str, torch.Tensor]]:
    &quot;&quot;&quot;Return CPU detection dictionaries in the manifest/dataset order, fitting nothing.&quot;&quot;&quot;
    model.to(device).eval()
    outputs = []
    for images, _ in loader:
        outputs.extend(
            {key: value.cpu() for key, value in prediction.items()}
            for prediction in model([image.to(device) for image in images])
        )
    return outputs


</code></pre>
</details>


### Worked component — `loader_for`

Run this supplied definition before its callers. Its inputs and invariants are explained in this
section.




In [ ]:
def loader_for(frame, training=False, labeled=True):
    if labeled:
        # Merge in manifest order; a box table's original order is not a prediction ID contract.
        ordered = frame[[ID_COLUMN]].merge(
            annotations, on=ID_COLUMN, sort=False, validate="one_to_many"
        )
        dataset = BoxDetectionDataset(ordered, root=DATA_DIR)
    else:
        dataset = DetectionInferenceDataset(frame, root=DATA_DIR)
    sampler = (
        WeightedRandomSampler(sampling_weights, len(dataset), replacement=True)
        if training and USE_IMBALANCE_WEIGHTS
        else None
    )
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=training and sampler is None,
        sampler=sampler,
        collate_fn=detection_collate,
        num_workers=0,
    )




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def loader_for(frame, training=False, labeled=True):
    if labeled:
        # Merge in manifest order; a box table&#x27;s original order is not a prediction ID contract.
        ordered = frame[[ID_COLUMN]].merge(
            annotations, on=ID_COLUMN, sort=False, validate=&quot;one_to_many&quot;
        )
        dataset = BoxDetectionDataset(ordered, root=DATA_DIR)
    else:
        dataset = DetectionInferenceDataset(frame, root=DATA_DIR)
    sampler = (
        WeightedRandomSampler(sampling_weights, len(dataset), replacement=True)
        if training and USE_IMBALANCE_WEIGHTS
        else None
    )
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=training and sampler is None,
        sampler=sampler,
        collate_fn=detection_collate,
        num_workers=0,
    )


</code></pre>
</details>


### Exercise 3 — `make_loader_pair`

Input: disjoint training/validation image records. Return two aligned loaders. Keep validation
deterministic; preserve per-image mask geometry or variable-length box targets.

<details>
<summary>Hint 1 — choose the operation</summary>

A dataset item returns an image tensor and a variable-length target dictionary. The collate
function returns lists, not stacked box arrays. Foreground labels start at 1 because torchvision
reserves 0 for background.

</details>

<details>
<summary>Hint 2 — implementation direction</summary>

Call the supplied loader_for for each fold. Detection trains with training=True; the first mask
sanity run uses deterministic geometry.

</details>




In [ ]:
def make_loader_pair(training_frame: pd.DataFrame, validation_frame: pd.DataFrame) -> Any:
    """Input: disjoint training/validation image records. Return two aligned loaders. Keep
    validation deterministic; preserve per-image mask geometry or variable-length box
    targets.
    """
    raise NotImplementedError('TODO: implement make_loader_pair using the stated contract')
    return train_loader, valid_loader




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def make_loader_pair(training_frame: pd.DataFrame, validation_frame: pd.DataFrame) -&gt; Any:
    &quot;&quot;&quot;Input: disjoint training/validation image records. Return two aligned loaders. Keep
    validation deterministic; preserve per-image mask geometry or variable-length box
    targets.
    &quot;&quot;&quot;
    train_loader, valid_loader = loader_for(training_frame, True), loader_for(validation_frame)
    return train_loader, valid_loader


</code></pre>
</details>


**Check before continuing.** Predict the result, run the small test below, and use its assertion
message to locate an error. Expected: all assertions pass and the printed success message
appears. These checks do not fit the real candidate.



In [ ]:
toy_train_loader, toy_valid_loader = make_loader_pair(tr, va)
toy_images, toy_targets = next(iter(toy_valid_loader))
assert len(toy_images) == len(toy_targets) and len(toy_images) > 0
print("Loader check passed: each image has its own aligned target.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>toy_train_loader, toy_valid_loader = make_loader_pair(tr, va)
toy_images, toy_targets = next(iter(toy_valid_loader))
assert len(toy_images) == len(toy_targets) and len(toy_images) &gt; 0
print(&quot;Loader check passed: each image has its own aligned target.&quot;)

</code></pre>
</details>


In [ ]:
train_loader, valid_loader = make_loader_pair(tr, va)
images, targets = next(iter(train_loader))
assert images[0].shape[0] == 3 and targets[0]["boxes"].shape[1] == 4
print("Images in batch:", len(images), "First target boxes:", targets[0]["boxes"].shape)
truth_boxes = [
    annotations.loc[annotations[ID_COLUMN] == key, ["xmin", "ymin", "xmax", "ymax"]].to_numpy()
    for key in va[ID_COLUMN]
]



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>train_loader, valid_loader = make_loader_pair(tr, va)
images, targets = next(iter(train_loader))
assert images[0].shape[0] == 3 and targets[0][&quot;boxes&quot;].shape[1] == 4
print(&quot;Images in batch:&quot;, len(images), &quot;First target boxes:&quot;, targets[0][&quot;boxes&quot;].shape)
truth_boxes = [
    annotations.loc[annotations[ID_COLUMN] == key, [&quot;xmin&quot;, &quot;ymin&quot;, &quot;xmax&quot;, &quot;ymax&quot;]].to_numpy()
    for key in va[ID_COLUMN]
]

</code></pre>
</details>


### Stop and explain

State the feature/batch shapes and the exact point at which preprocessing is fitted. What
happens to a previously unseen validation input?

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

A dataset item returns an image tensor and a variable-length target dictionary. The collate
function returns lists, not stacked box arrays. Foreground labels start at 1 because torchvision
reserves 0 for background.

</details>



## 7. Connect outputs, loss and official metric

Faster R-CNN with a MobileNet backbone is the first delivery route. Its head has two classes
including background. Both detector and backbone weight downloads stay disabled unless a
permitted local checkpoint is supplied.



### Supplied helper — `save_torch_state`

Atomically save torch state without importing torch until this function is used.




In [ ]:
def save_torch_state(value: Any, path: Path | str) -> Path:
    """Atomically save torch state without importing torch until this function is used."""
    return _write(value, path, "torch")




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def save_torch_state(value: Any, path: Path | str) -&gt; Path:
    &quot;&quot;&quot;Atomically save torch state without importing torch until this function is used.&quot;&quot;&quot;
    return _write(value, path, &quot;torch&quot;)


</code></pre>
</details>


### Supplied helper — `sync_artifacts`

Copy the entire candidate, including encoders, preserving separate run folders.




In [ ]:
def sync_artifacts(source: Path | str, destination: Path | str | None) -> list[Path]:
    """Copy the entire candidate, including encoders, preserving separate run folders."""
    if destination is None:
        return []
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if (
        destination == source
        or destination.is_relative_to(source)
        or source.is_relative_to(destination)
    ):
        raise ValueError("Output and backup folders must be separate")
    copied = []
    for path in source.rglob("*"):
        if path.is_file() and not path.name.endswith(".tmp"):
            target = destination / path.relative_to(source)
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_name(f".{target.name}.{uuid4().hex}.tmp")
            try:
                shutil.copy2(path, temporary)
                temporary.replace(target)
            finally:
                temporary.unlink(missing_ok=True)
            copied.append(target)
    return copied




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def sync_artifacts(source: Path | str, destination: Path | str | None) -&gt; list[Path]:
    &quot;&quot;&quot;Copy the entire candidate, including encoders, preserving separate run folders.&quot;&quot;&quot;
    if destination is None:
        return []
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if (
        destination == source
        or destination.is_relative_to(source)
        or source.is_relative_to(destination)
    ):
        raise ValueError(&quot;Output and backup folders must be separate&quot;)
    copied = []
    for path in source.rglob(&quot;*&quot;):
        if path.is_file() and not path.name.endswith(&quot;.tmp&quot;):
            target = destination / path.relative_to(source)
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_name(f&quot;.{target.name}.{uuid4().hex}.tmp&quot;)
            try:
                shutil.copy2(path, temporary)
                temporary.replace(target)
            finally:
                temporary.unlink(missing_ok=True)
            copied.append(target)
    return copied


</code></pre>
</details>


### Worked component — `local_checkpoint`

Resolve a staged checkpoint; pretrained opt-in requires an explicit permitted local file.

A checkpoint supplied with pretrained disabled can be the team's own trained state. The
caller must record provenance; a local path alone does not establish competition permission.




In [ ]:
def local_checkpoint(pretrained_allowed: bool, checkpoint_path: Path | str | None) -> str:
    """Resolve a staged checkpoint; pretrained opt-in requires an explicit permitted local file.

    A checkpoint supplied with pretrained disabled can be the team's own trained state. The
    caller must record provenance; a local path alone does not establish competition permission.
    """
    if pretrained_allowed and checkpoint_path is None:
        raise ValueError("Stage the exact permitted weights locally and set checkpoint_path")
    if checkpoint_path is None:
        return ""
    path = Path(checkpoint_path)
    if not path.is_file():
        raise FileNotFoundError(path)
    return str(path)




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def local_checkpoint(pretrained_allowed: bool, checkpoint_path: Path | str | None) -&gt; str:
    &quot;&quot;&quot;Resolve a staged checkpoint; pretrained opt-in requires an explicit permitted local file.

    A checkpoint supplied with pretrained disabled can be the team&#x27;s own trained state. The
    caller must record provenance; a local path alone does not establish competition permission.
    &quot;&quot;&quot;
    if pretrained_allowed and checkpoint_path is None:
        raise ValueError(&quot;Stage the exact permitted weights locally and set checkpoint_path&quot;)
    if checkpoint_path is None:
        return &quot;&quot;
    path = Path(checkpoint_path)
    if not path.is_file():
        raise FileNotFoundError(path)
    return str(path)


</code></pre>
</details>


### Worked component — `build_detection_model`

Build a torchvision box detector and replace its classification head.

Args:
    num_classes: Total number of labels including background class 0.
    architecture: ``fasterrcnn_resnet50_fpn_v2``, ``fasterrcnn_resnet50_fpn``, or
        ``fasterrcnn_mobilenet_v3_large_fpn``.
    pretrained_allowed: Require explicitly staged permitted weights when enabled.
    checkpoint_path: Original permitted COCO state for initialization; matching task state
        for reload.




In [ ]:
def build_detection_model(
    num_classes: int,
    *,
    architecture: str = "fasterrcnn_resnet50_fpn_v2",
    pretrained_allowed: bool = False,
    checkpoint_path: Path | str | None = None,
) -> nn.Module:
    """Build a torchvision box detector and replace its classification head.

    Args:
        num_classes: Total number of labels including background class 0.
        architecture: ``fasterrcnn_resnet50_fpn_v2``, ``fasterrcnn_resnet50_fpn``, or
            ``fasterrcnn_mobilenet_v3_large_fpn``.
        pretrained_allowed: Require explicitly staged permitted weights when enabled.
        checkpoint_path: Original permitted COCO state for initialization; matching task state
            for reload.
    """
    from torchvision.models.detection import (
        fasterrcnn_mobilenet_v3_large_fpn,
        fasterrcnn_resnet50_fpn,
        fasterrcnn_resnet50_fpn_v2,
    )
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

    factories = {
        "fasterrcnn_resnet50_fpn_v2": fasterrcnn_resnet50_fpn_v2,
        "fasterrcnn_resnet50_fpn": fasterrcnn_resnet50_fpn,
        "fasterrcnn_mobilenet_v3_large_fpn": fasterrcnn_mobilenet_v3_large_fpn,
    }
    try:
        factory = factories[architecture]
    except KeyError as error:
        raise ValueError(
            f"Unknown detector {architecture!r}; choose {sorted(factories)}"
        ) from error
    checkpoint = local_checkpoint(pretrained_allowed, checkpoint_path)
    model = factory(weights=None, weights_backbone=None)
    if checkpoint and pretrained_allowed:
        model.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True))
    features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(features, num_classes)
    if checkpoint and not pretrained_allowed:
        model.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True))
    return model




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def build_detection_model(
    num_classes: int,
    *,
    architecture: str = &quot;fasterrcnn_resnet50_fpn_v2&quot;,
    pretrained_allowed: bool = False,
    checkpoint_path: Path | str | None = None,
) -&gt; nn.Module:
    &quot;&quot;&quot;Build a torchvision box detector and replace its classification head.

    Args:
        num_classes: Total number of labels including background class 0.
        architecture: ``fasterrcnn_resnet50_fpn_v2``, ``fasterrcnn_resnet50_fpn``, or
            ``fasterrcnn_mobilenet_v3_large_fpn``.
        pretrained_allowed: Require explicitly staged permitted weights when enabled.
        checkpoint_path: Original permitted COCO state for initialization; matching task state
            for reload.
    &quot;&quot;&quot;
    from torchvision.models.detection import (
        fasterrcnn_mobilenet_v3_large_fpn,
        fasterrcnn_resnet50_fpn,
        fasterrcnn_resnet50_fpn_v2,
    )
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

    factories = {
        &quot;fasterrcnn_resnet50_fpn_v2&quot;: fasterrcnn_resnet50_fpn_v2,
        &quot;fasterrcnn_resnet50_fpn&quot;: fasterrcnn_resnet50_fpn,
        &quot;fasterrcnn_mobilenet_v3_large_fpn&quot;: fasterrcnn_mobilenet_v3_large_fpn,
    }
    try:
        factory = factories[architecture]
    except KeyError as error:
        raise ValueError(
            f&quot;Unknown detector {architecture!r}; choose {sorted(factories)}&quot;
        ) from error
    checkpoint = local_checkpoint(pretrained_allowed, checkpoint_path)
    model = factory(weights=None, weights_backbone=None)
    if checkpoint and pretrained_allowed:
        model.load_state_dict(torch.load(checkpoint, map_location=&quot;cpu&quot;, weights_only=True))
    features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(features, num_classes)
    if checkpoint and not pretrained_allowed:
        model.load_state_dict(torch.load(checkpoint, map_location=&quot;cpu&quot;, weights_only=True))
    return model


</code></pre>
</details>


### Worked component — `train_detection_epoch`

Train one epoch using the loss dictionary returned by torchvision detectors.




In [ ]:
def train_detection_epoch(
    model: nn.Module,
    loader: Any,
    optimizer: torch.optim.Optimizer,
    *,
    device: str = "auto",
    time_budget: TimeBudget | None = None,
) -> float:
    """Train one epoch using the loss dictionary returned by torchvision detectors."""
    resolved = torch.device(
        device if device != "auto" else "cuda" if torch.cuda.is_available() else "cpu"
    )
    model.to(resolved).train()
    total = 0.0
    batches = 0
    for images, targets in loader:
        if time_budget and time_budget.should_stop:
            break
        images = [image.to(resolved) for image in images]
        targets = [{key: value.to(resolved) for key, value in target.items()} for target in targets]
        losses = model(images, targets)
        loss = sum(losses.values())
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total += float(loss.detach())
        batches += 1
    return total / max(batches, 1)




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def train_detection_epoch(
    model: nn.Module,
    loader: Any,
    optimizer: torch.optim.Optimizer,
    *,
    device: str = &quot;auto&quot;,
    time_budget: TimeBudget | None = None,
) -&gt; float:
    &quot;&quot;&quot;Train one epoch using the loss dictionary returned by torchvision detectors.&quot;&quot;&quot;
    resolved = torch.device(
        device if device != &quot;auto&quot; else &quot;cuda&quot; if torch.cuda.is_available() else &quot;cpu&quot;
    )
    model.to(resolved).train()
    total = 0.0
    batches = 0
    for images, targets in loader:
        if time_budget and time_budget.should_stop:
            break
        images = [image.to(resolved) for image in images]
        targets = [{key: value.to(resolved) for key, value in target.items()} for target in targets]
        losses = model(images, targets)
        loss = sum(losses.values())
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total += float(loss.detach())
        batches += 1
    return total / max(batches, 1)


</code></pre>
</details>


<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>
</code></pre>
</details>


### Worked component — `wheat_score`

Wheat reference convention: confidence-ordered greedy matching, mean .50:.05:.75.

Absolute xyxy boxes; the linked winning implementation uses inclusive (+1) pixel areas.
Expose that choice because other graders use continuous exclusive coordinates. Compare
fixtures with the actual official grader before transferring this metric to a new contest.
The task statement uses strict IoU comparisons and scores every no-target image zero.
Set strict_iou=False only to compare with the linked winner's non-strict implementation.
This averages TP/(TP+FP+FN) over thresholds then images, not confidence-integrated AP.




In [ ]:
def wheat_score(
    targets: Sequence[ArrayLike],
    boxes: Sequence[ArrayLike],
    scores: Sequence[ArrayLike],
    *,
    score_threshold: float = 0.5,
    inclusive_pixels: bool = True,
    strict_iou: bool = True,
) -> float:
    """Wheat reference convention: confidence-ordered greedy matching, mean .50:.05:.75.

    Absolute xyxy boxes; the linked winning implementation uses inclusive (+1) pixel areas.
    Expose that choice because other graders use continuous exclusive coordinates. Compare
    fixtures with the actual official grader before transferring this metric to a new contest.
    The task statement uses strict IoU comparisons and scores every no-target image zero.
    Set strict_iou=False only to compare with the linked winner's non-strict implementation.
    This averages TP/(TP+FP+FN) over thresholds then images, not confidence-integrated AP.
    """
    if not len(targets) or not (len(targets) == len(boxes) == len(scores)):
        raise ValueError("Need one target, prediction and confidence array per image")
    image_scores = []
    offset = 1.0 if inclusive_pixels else 0.0
    for references, predictions, confidences in zip(targets, boxes, scores, strict=True):
        gt = np.asarray(references, dtype=float).reshape(-1, 4)
        predicted = np.asarray(predictions, dtype=float).reshape(-1, 4)
        confidence = np.asarray(confidences, dtype=float)
        if confidence.shape != (len(predicted),) or not np.isfinite(confidence).all():
            raise ValueError("One finite confidence per predicted box is required")
        for rectangles in (gt, predicted):
            if not np.isfinite(rectangles).all() or np.any(rectangles[:, 2:] <= rectangles[:, :2]):
                raise ValueError("Boxes must be finite positive-area xyxy rectangles")
        if not len(gt):
            image_scores.append(0.0)
            continue
        keep = np.argsort(-confidence, kind="stable")
        predicted = predicted[keep[confidence[keep] > score_threshold]]
        low = np.maximum(predicted[:, None, :2], gt[None, :, :2])
        high = np.minimum(predicted[:, None, 2:], gt[None, :, 2:])
        intersection = np.maximum(high - low + offset, 0).prod(axis=2)
        area_p = (predicted[:, 2:] - predicted[:, :2] + offset).prod(axis=1)
        area_y = (gt[:, 2:] - gt[:, :2] + offset).prod(axis=1)
        iou = intersection / np.maximum(area_p[:, None] + area_y[None, :] - intersection, 1e-12)
        threshold_scores = []
        for threshold in np.arange(50, 80, 5) / 100:
            available = np.ones(len(gt), dtype=bool)
            matched = 0
            for row in iou:
                matched_iou = row > threshold if strict_iou else row >= threshold
                candidates = np.where(available & matched_iou)[0]
                if len(candidates):
                    best = candidates[np.argmax(row[candidates])]
                    available[best] = False
                    matched += 1
            denominator = len(gt) + len(predicted) - matched
            threshold_scores.append(matched / denominator if denominator else 1.0)
        image_scores.append(np.mean(threshold_scores))
    return float(np.mean(image_scores))




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def wheat_score(
    targets: Sequence[ArrayLike],
    boxes: Sequence[ArrayLike],
    scores: Sequence[ArrayLike],
    *,
    score_threshold: float = 0.5,
    inclusive_pixels: bool = True,
    strict_iou: bool = True,
) -&gt; float:
    &quot;&quot;&quot;Wheat reference convention: confidence-ordered greedy matching, mean .50:.05:.75.

    Absolute xyxy boxes; the linked winning implementation uses inclusive (+1) pixel areas.
    Expose that choice because other graders use continuous exclusive coordinates. Compare
    fixtures with the actual official grader before transferring this metric to a new contest.
    The task statement uses strict IoU comparisons and scores every no-target image zero.
    Set strict_iou=False only to compare with the linked winner&#x27;s non-strict implementation.
    This averages TP/(TP+FP+FN) over thresholds then images, not confidence-integrated AP.
    &quot;&quot;&quot;
    if not len(targets) or not (len(targets) == len(boxes) == len(scores)):
        raise ValueError(&quot;Need one target, prediction and confidence array per image&quot;)
    image_scores = []
    offset = 1.0 if inclusive_pixels else 0.0
    for references, predictions, confidences in zip(targets, boxes, scores, strict=True):
        gt = np.asarray(references, dtype=float).reshape(-1, 4)
        predicted = np.asarray(predictions, dtype=float).reshape(-1, 4)
        confidence = np.asarray(confidences, dtype=float)
        if confidence.shape != (len(predicted),) or not np.isfinite(confidence).all():
            raise ValueError(&quot;One finite confidence per predicted box is required&quot;)
        for rectangles in (gt, predicted):
            if not np.isfinite(rectangles).all() or np.any(rectangles[:, 2:] &lt;= rectangles[:, :2]):
                raise ValueError(&quot;Boxes must be finite positive-area xyxy rectangles&quot;)
        if not len(gt):
            image_scores.append(0.0)
            continue
        keep = np.argsort(-confidence, kind=&quot;stable&quot;)
        predicted = predicted[keep[confidence[keep] &gt; score_threshold]]
        low = np.maximum(predicted[:, None, :2], gt[None, :, :2])
        high = np.minimum(predicted[:, None, 2:], gt[None, :, 2:])
        intersection = np.maximum(high - low + offset, 0).prod(axis=2)
        area_p = (predicted[:, 2:] - predicted[:, :2] + offset).prod(axis=1)
        area_y = (gt[:, 2:] - gt[:, :2] + offset).prod(axis=1)
        iou = intersection / np.maximum(area_p[:, None] + area_y[None, :] - intersection, 1e-12)
        threshold_scores = []
        for threshold in np.arange(50, 80, 5) / 100:
            available = np.ones(len(gt), dtype=bool)
            matched = 0
            for row in iou:
                matched_iou = row &gt; threshold if strict_iou else row &gt;= threshold
                candidates = np.where(available &amp; matched_iou)[0]
                if len(candidates):
                    best = candidates[np.argmax(row[candidates])]
                    available[best] = False
                    matched += 1
            denominator = len(gt) + len(predicted) - matched
            threshold_scores.append(matched / denominator if denominator else 1.0)
        image_scores.append(np.mean(threshold_scores))
    return float(np.mean(image_scores))


</code></pre>
</details>


In [ ]:
if PRETRAINED_ALLOWED and not ASSET_PROVENANCE:
    raise ValueError("Record the exact permitted local detector weights")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>if PRETRAINED_ALLOWED and not ASSET_PROVENANCE:
    raise ValueError(&quot;Record the exact permitted local detector weights&quot;)

</code></pre>
</details>


### Exercise 4 — `make_model`

Construct the stated scratch architecture with the exact output dimensions and legal weight
policy. The actual batch/model compatibility check below must pass before any full training.

<details>
<summary>Hint 1 — choose the operation</summary>

Faster R-CNN with a MobileNet backbone is the first delivery route. Its head has two classes
including background. Both detector and backbone weight downloads stay disabled unless a
permitted local checkpoint is supplied.

</details>

<details>
<summary>Hint 2 — implementation direction</summary>

include background and disable all implicit weight downloads

</details>




In [ ]:
def make_model(initialize=False):
    raise NotImplementedError('TODO: include background and disable all implicit weight downloads')
    network.transform.min_size = (IMAGE_SIZE,)
    network.transform.max_size = IMAGE_SIZE
    return network




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def make_model(initialize=False):
    network = build_detection_model(
        num_classes=2,
        architecture=MODEL_NAME,
        pretrained_allowed=PRETRAINED_ALLOWED if initialize else False,
        checkpoint_path=WEIGHT_PATH if initialize else None,
    )
    network.transform.min_size = (IMAGE_SIZE,)
    network.transform.max_size = IMAGE_SIZE
    return network


</code></pre>
</details>


In [ ]:
model = make_model(initialize=True).to(competition.device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
MODEL_DESCRIPTION = MODEL_NAME



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>model = make_model(initialize=True).to(competition.device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
MODEL_DESCRIPTION = MODEL_NAME

</code></pre>
</details>


### Worked component — `evaluate_boxes`

Run this supplied definition before its callers. Its inputs and invariants are explained in this
section.




In [ ]:
def evaluate_boxes(predictions, threshold):
    return wheat_score(
        truth_boxes,
        [item["boxes"].numpy() for item in predictions],
        [item["scores"].numpy() for item in predictions],
        score_threshold=threshold,
        inclusive_pixels=True,
    )




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def evaluate_boxes(predictions, threshold):
    return wheat_score(
        truth_boxes,
        [item[&quot;boxes&quot;].numpy() for item in predictions],
        [item[&quot;scores&quot;].numpy() for item in predictions],
        score_threshold=threshold,
        inclusive_pixels=True,
    )


</code></pre>
</details>


### Exercise 5 — `keep_best_candidate`

Input: complete validation score, best score so far, model and destination. Return the updated
best score; atomically save only an improving model. Inferior epochs must leave the previous
bytes unchanged.

<details>
<summary>Hint 1 — choose the operation</summary>

Use the detector's FP32 loss dictionary and preserve the best validation-selected state
atomically. An inferior epoch must not overwrite it. Reserve time for full-image inference and
prediction-string validation.

</details>

<details>
<summary>Hint 2 — implementation direction</summary>

Compare scores before saving; preserve model state, epoch and score together, then return the
selected best value.

</details>




In [ ]:
def keep_best_candidate(
    value: float, best: float, network: nn.Module, epoch: int, checkpoint_path: Path
) -> Any:
    """Input: complete validation score, best score so far, model and destination. Return the
    updated best score; atomically save only an improving model. Inferior epochs must leave
    the previous bytes unchanged.
    """
    raise NotImplementedError('TODO: implement keep_best_candidate using the stated contract')
    return best




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def keep_best_candidate(
    value: float, best: float, network: nn.Module, epoch: int, checkpoint_path: Path
) -&gt; Any:
    &quot;&quot;&quot;Input: complete validation score, best score so far, model and destination. Return the
    updated best score; atomically save only an improving model. Inferior epochs must leave
    the previous bytes unchanged.
    &quot;&quot;&quot;
    if value &gt; best:
        best = value
        save_torch_state(
            {&quot;model&quot;: network.state_dict(), &quot;epoch&quot;: epoch, &quot;score&quot;: value}, checkpoint_path
        )
    return best


</code></pre>
</details>


**Check before continuing.** Predict the result, run the small test below, and use its assertion
message to locate an error. Expected: all assertions pass and the printed success message
appears. These checks do not fit the real candidate.



In [ ]:
with torch.random.fork_rng(devices=[]):
    toy_network = nn.Linear(2, 1)
    toy_path = OUTPUT / "checks" / "best-policy.pt"
    toy_best = keep_best_candidate(0.7, -float("inf"), toy_network, 0, toy_path)
    toy_bytes = toy_path.read_bytes()
    assert toy_best == 0.7
    assert keep_best_candidate(0.2, toy_best, toy_network, 1, toy_path) == 0.7
    assert toy_path.read_bytes() == toy_bytes, "An inferior score overwrote the best checkpoint"
print("Checkpoint checks passed: worse epochs do not replace the selected candidate.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>with torch.random.fork_rng(devices=[]):
    toy_network = nn.Linear(2, 1)
    toy_path = OUTPUT / &quot;checks&quot; / &quot;best-policy.pt&quot;
    toy_best = keep_best_candidate(0.7, -float(&quot;inf&quot;), toy_network, 0, toy_path)
    toy_bytes = toy_path.read_bytes()
    assert toy_best == 0.7
    assert keep_best_candidate(0.2, toy_best, toy_network, 1, toy_path) == 0.7
    assert toy_path.read_bytes() == toy_bytes, &quot;An inferior score overwrote the best checkpoint&quot;
print(&quot;Checkpoint checks passed: worse epochs do not replace the selected candidate.&quot;)

</code></pre>
</details>


### Exercise 6 — `select_confidence`

Input: validation box dictionaries, confidence grid and official scoring callback. Return
scores, selected confidence and score. The callback owns the image truth and IoU convention.

<details>
<summary>Hint 1 — choose the operation</summary>

Sweep a small confidence grid on held-out images only with the actual Wheat scoring convention.
Duplicate boxes count as false positives. Confidence filtering is distinct from the IoU
criterion for matching boxes.

</details>

<details>
<summary>Hint 2 — implementation direction</summary>

Score every confidence on the same validation predictions; maximize score, breaking ties toward
0.5.

</details>




In [ ]:
def select_confidence(
    predictions: Sequence[dict], candidates: Sequence[float], score_fn: Any
) -> Any:
    """Input: validation box dictionaries, confidence grid and official scoring callback.
    Return scores, selected confidence and score. The callback owns the image truth and IoU
    convention.
    """
    raise NotImplementedError('TODO: implement select_confidence using the stated contract')
    return scores_by_threshold, selected_threshold, validation_score




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def select_confidence(
    predictions: Sequence[dict], candidates: Sequence[float], score_fn: Any
) -&gt; Any:
    &quot;&quot;&quot;Input: validation box dictionaries, confidence grid and official scoring callback.
    Return scores, selected confidence and score. The callback owns the image truth and IoU
    convention.
    &quot;&quot;&quot;
    scores_by_threshold = {threshold: score_fn(predictions, threshold) for threshold in candidates}
    selected_threshold = max(
        candidates, key=lambda value: (scores_by_threshold[value], -abs(value - 0.5))
    )
    validation_score = scores_by_threshold[selected_threshold]
    return scores_by_threshold, selected_threshold, validation_score


</code></pre>
</details>


**Check before continuing.** Predict the result, run the small test below, and use its assertion
message to locate an error. Expected: all assertions pass and the printed success message
appears. These checks do not fit the real candidate.



In [ ]:
toy_predictions = [{"boxes": np.empty((0, 4)), "scores": np.array([])}]
toy_scores, toy_threshold, toy_score = select_confidence(
    toy_predictions, (0.2, 0.5, 0.8), lambda predictions, threshold: 1 - abs(threshold - 0.5)
)
assert toy_threshold == 0.5 and toy_score == 1
assert len(toy_scores) == 3
print("Confidence checks passed: selection uses the supplied validation scorer.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>toy_predictions = [{&quot;boxes&quot;: np.empty((0, 4)), &quot;scores&quot;: np.array([])}]
toy_scores, toy_threshold, toy_score = select_confidence(
    toy_predictions, (0.2, 0.5, 0.8), lambda predictions, threshold: 1 - abs(threshold - 0.5)
)
assert toy_threshold == 0.5 and toy_score == 1
assert len(toy_scores) == 3
print(&quot;Confidence checks passed: selection uses the supplied validation scorer.&quot;)

</code></pre>
</details>


### Stop and explain

Explain the output shape, loss and prediction decoding without reading the implementation. Which
convention would silently produce an invalid score?

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

Faster R-CNN with a MobileNet backbone is the first delivery route. Its head has two classes
including background. Both detector and backbone weight downloads stay disabled unless a
permitted local checkpoint is supplied. Sweep a small confidence grid on held-out images only
with the actual Wheat scoring convention. Duplicate boxes count as false positives. Confidence
filtering is distinct from the IoU criterion for matching boxes.

</details>



## 8. Train and preserve a recoverable candidate

Use the detector's FP32 loss dictionary and preserve the best validation-selected state
atomically. An inferior epoch must not overwrite it. Reserve time for full-image inference and
prediction-string validation.



In [ ]:
started = perf_counter()
best = -float("inf")
checkpoint_path = OUTPUT / "best.pt"
for epoch in range(EPOCHS):
    if budget.should_stop:
        break
    loss = train_detection_epoch(
        model, train_loader, optimizer, device=competition.device, time_budget=budget
    )
    if budget.should_stop:
        break
    predictions = predict_detections(model, valid_loader, device=competition.device)
    value = evaluate_boxes(predictions, 0.5)
    best = keep_best_candidate(value, best, model, epoch, checkpoint_path)
    print(epoch, "loss", loss, "wheat reference score", value)
if not checkpoint_path.exists():
    raise TimeoutError("No complete validation before the reserve; shorten the debug candidate")
model.load_state_dict(
    torch.load(checkpoint_path, map_location=competition.device, weights_only=True)["model"]
)
fit_seconds = perf_counter() - started
validation_predictions = predict_detections(model, valid_loader, device=competition.device)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>started = perf_counter()
best = -float(&quot;inf&quot;)
checkpoint_path = OUTPUT / &quot;best.pt&quot;
for epoch in range(EPOCHS):
    if budget.should_stop:
        break
    loss = train_detection_epoch(
        model, train_loader, optimizer, device=competition.device, time_budget=budget
    )
    if budget.should_stop:
        break
    predictions = predict_detections(model, valid_loader, device=competition.device)
    value = evaluate_boxes(predictions, 0.5)
    best = keep_best_candidate(value, best, model, epoch, checkpoint_path)
    print(epoch, &quot;loss&quot;, loss, &quot;wheat reference score&quot;, value)
if not checkpoint_path.exists():
    raise TimeoutError(&quot;No complete validation before the reserve; shorten the debug candidate&quot;)
model.load_state_dict(
    torch.load(checkpoint_path, map_location=competition.device, weights_only=True)[&quot;model&quot;]
)
fit_seconds = perf_counter() - started
validation_predictions = predict_detections(model, valid_loader, device=competition.device)

</code></pre>
</details>


In [ ]:
assert checkpoint_path.is_file(), "No saved candidate is available for inference"
assert np.isfinite(fit_seconds) and fit_seconds >= 0
print("Candidate checks passed: persisted artifact and measured fit time.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>assert checkpoint_path.is_file(), &quot;No saved candidate is available for inference&quot;
assert np.isfinite(fit_seconds) and fit_seconds &gt;= 0
print(&quot;Candidate checks passed: persisted artifact and measured fit time.&quot;)

</code></pre>
</details>


### Stop and explain

What evidence proves the saved artifact can reproduce inference? Which training state is not
promised by this reload?

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

Keep architecture/preprocessing, label order and best weights together, then compare reloaded
validation predictions. This lesson preserves inference recovery, not an exact continuation of
optimizer/scaler/random state.

</details>



## 9. Interpret validation and compare experiments

Sweep a small confidence grid on held-out images only with the actual Wheat scoring convention.
Duplicate boxes count as false positives. Confidence filtering is distinct from the IoU
criterion for matching boxes.



In [ ]:
candidates = (0.5,)
scores_by_threshold, selected_threshold, validation_score = select_confidence(
    validation_predictions, candidates, evaluate_boxes
)
print("Wheat reference score:", validation_score, scores_by_threshold)
print("No-box reference under the statement: 0.0")
errors = pd.DataFrame(
    {
        ID_COLUMN: va[ID_COLUMN],
        "true_boxes": [len(value) for value in truth_boxes],
        "predicted_boxes": [
            int((value["scores"] > selected_threshold).sum()) for value in validation_predictions
        ],
    }
)
print(errors.to_string(index=False))
errors.to_csv(OUTPUT / "validation_errors.csv", index=False)
plt.scatter(errors["true_boxes"], errors["predicted_boxes"])
plt.xlabel("True box count")
plt.ylabel("Predicted box count")
plt.savefig(OUTPUT / "validation.png")
plt.show()
plt.close()



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>candidates = (0.5,)
scores_by_threshold, selected_threshold, validation_score = select_confidence(
    validation_predictions, candidates, evaluate_boxes
)
print(&quot;Wheat reference score:&quot;, validation_score, scores_by_threshold)
print(&quot;No-box reference under the statement: 0.0&quot;)
errors = pd.DataFrame(
    {
        ID_COLUMN: va[ID_COLUMN],
        &quot;true_boxes&quot;: [len(value) for value in truth_boxes],
        &quot;predicted_boxes&quot;: [
            int((value[&quot;scores&quot;] &gt; selected_threshold).sum()) for value in validation_predictions
        ],
    }
)
print(errors.to_string(index=False))
errors.to_csv(OUTPUT / &quot;validation_errors.csv&quot;, index=False)
plt.scatter(errors[&quot;true_boxes&quot;], errors[&quot;predicted_boxes&quot;])
plt.xlabel(&quot;True box count&quot;)
plt.ylabel(&quot;Predicted box count&quot;)
plt.savefig(OUTPUT / &quot;validation.png&quot;)
plt.show()
plt.close()

</code></pre>
</details>


In [ ]:
assert np.isfinite(validation_score), "Selection requires a finite official score"
print("Validation check passed. Compare this score and runtime with the baseline ledger.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>assert np.isfinite(validation_score), &quot;Selection requires a finite official score&quot;
print(&quot;Validation check passed. Compare this score and runtime with the baseline ledger.&quot;)

</code></pre>
</details>


### Stop and explain

What is the next single experiment, and what result would make you reject it?

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

Compare MobileNet and ResNet-FPN using the same image/source split and grader. Include full test
inference time. Test resolution or confidence independently; do not introduce an ensemble before
measuring its delivery cost.

</details>



## 10. Reload and infer every test item

Rebuild the detector without redownloading initialization, load its best state and reproduce
validation boxes. Predict all test image records, retain their identities and convert final xyxy
boxes to output xywh.



In [ ]:
started = perf_counter()



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>started = perf_counter()

</code></pre>
</details>


### Exercise 7 — Reload and predict

Rebuild the detector without redownloading initialization, load its best state and reproduce
validation boxes. Predict all test image records, retain their identities and convert final xyxy
boxes to output xywh.

<details>
<summary>Hint 1 — choose the operation</summary>

Rebuild the detector without redownloading initialization, load its best state and reproduce
validation boxes. Predict all test image records, retain their identities and convert final xyxy
boxes to output xywh.

</details>

<details>
<summary>Hint 2 — implementation direction</summary>

rebuild the detector and reload before complete test inference

</details>



In [ ]:
raise NotImplementedError('TODO: rebuild the detector and reload before complete test inference')



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>restored = make_model().to(competition.device)
restored.load_state_dict(
    torch.load(checkpoint_path, map_location=competition.device, weights_only=True)[&quot;model&quot;],
    strict=True,
)
reloaded = predict_detections(restored, valid_loader, device=competition.device)
for expected, actual in zip(validation_predictions, reloaded, strict=True):
    np.testing.assert_allclose(expected[&quot;boxes&quot;], actual[&quot;boxes&quot;], rtol=1e-5, atol=1e-5)
test_predictions = predict_detections(
    restored, loader_for(test, labeled=False), device=competition.device
)

</code></pre>
</details>


### Worked component — `prediction_string`

Run this supplied definition before its callers. Its inputs and invariants are explained in this
section.




In [ ]:
def prediction_string(prediction):
    values = []
    for score, box in zip(prediction["scores"], prediction["boxes"], strict=True):
        if float(score) <= selected_threshold:
            continue
        x1, y1, x2, y2 = box.tolist()
        if not np.isfinite([float(score), x1, y1, x2, y2]).all() or x2 <= x1 or y2 <= y1:
            raise ValueError("Invalid predicted box")
        values.append(f"{float(score):.6f} {x1:.4f} {y1:.4f} {x2 - x1:.4f} {y2 - y1:.4f}")
    return " ".join(values)




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def prediction_string(prediction):
    values = []
    for score, box in zip(prediction[&quot;scores&quot;], prediction[&quot;boxes&quot;], strict=True):
        if float(score) &lt;= selected_threshold:
            continue
        x1, y1, x2, y2 = box.tolist()
        if not np.isfinite([float(score), x1, y1, x2, y2]).all() or x2 &lt;= x1 or y2 &lt;= y1:
            raise ValueError(&quot;Invalid predicted box&quot;)
        values.append(f&quot;{float(score):.6f} {x1:.4f} {y1:.4f} {x2 - x1:.4f} {y2 - y1:.4f}&quot;)
    return &quot; &quot;.join(values)


</code></pre>
</details>


In [ ]:
output_values = {"PredictionString": [prediction_string(value) for value in test_predictions]}
save_json(
    {
        "score_threshold": selected_threshold,
        "resize": IMAGE_SIZE,
        "architecture": MODEL_NAME,
        "box_input": "xyxy",
        "submission": "score x y width height",
        "metric": "confidence-ordered inclusive-pixel wheat reference; not COCO AP",
    },
    OUTPUT / "recipe.json",
)
inference_seconds = perf_counter() - started



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>output_values = {&quot;PredictionString&quot;: [prediction_string(value) for value in test_predictions]}
save_json(
    {
        &quot;score_threshold&quot;: selected_threshold,
        &quot;resize&quot;: IMAGE_SIZE,
        &quot;architecture&quot;: MODEL_NAME,
        &quot;box_input&quot;: &quot;xyxy&quot;,
        &quot;submission&quot;: &quot;score x y width height&quot;,
        &quot;metric&quot;: &quot;confidence-ordered inclusive-pixel wheat reference; not COCO AP&quot;,
    },
    OUTPUT / &quot;recipe.json&quot;,
)
inference_seconds = perf_counter() - started

</code></pre>
</details>


In [ ]:
assert set(output_values) == set(sample.columns) - {ID_COLUMN}
assert all(len(values) == len(test) for values in output_values.values())
print("Inference checks passed: every test item has the required output columns.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>assert set(output_values) == set(sample.columns) - {ID_COLUMN}
assert all(len(values) == len(test) for values in output_values.values())
print(&quot;Inference checks passed: every test item has the required output columns.&quot;)

</code></pre>
</details>


### Stop and explain

Can this saved candidate handle every private input without refitting or downloading anything?
Which assets must travel with it?

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

Rebuild the detector without redownloading initialization, load its best state and reproduce
validation boxes. Predict all test image records, retain their identities and convert final xyxy
boxes to output xywh.

</details>



## 11. Deliver and verify the submission

One image may contain many boxes, so annotation rows must first be grouped by image_id. The
output is a prediction string of confidence and xywh boxes per image, including a valid empty
string when no box survives.

Write the CSV, read it back, and verify IDs, columns, row count, missing values and allowed
prediction values before saving the complete recipe. A local check does not confirm organizer
upload acceptance.



### Supplied helper — `SubmissionValidation`

Validation errors and non-fatal warnings for a candidate submission.




In [ ]:
@dataclass(slots=True)
class SubmissionValidation:
    """Validation errors and non-fatal warnings for a candidate submission."""

    valid: bool
    errors: list[str] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)

    def raise_for_errors(self) -> None:
        """Raise one ``ValueError`` containing every validation error."""
        if self.errors:
            raise ValueError("Invalid submission: " + "; ".join(self.errors))




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>@dataclass(slots=True)
class SubmissionValidation:
    &quot;&quot;&quot;Validation errors and non-fatal warnings for a candidate submission.&quot;&quot;&quot;

    valid: bool
    errors: list[str] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)

    def raise_for_errors(self) -&gt; None:
        &quot;&quot;&quot;Raise one ``ValueError`` containing every validation error.&quot;&quot;&quot;
        if self.errors:
            raise ValueError(&quot;Invalid submission: &quot; + &quot;; &quot;.join(self.errors))


</code></pre>
</details>


### Supplied helper — `build_submission`

Copy the sample and optionally align predictions using unique (possibly composite) IDs.

Without IDs, predictions must already be in sample order. Prefer explicit IDs for inference.
Alignment rejects missing, duplicate, extra, or absent keys rather than guessing row order.




In [ ]:
def build_submission(
    sample: pd.DataFrame,
    predictions: Mapping[str, Sequence[object]],
    *,
    prediction_ids: pd.DataFrame | Sequence[object] | None = None,
    id_columns: str | Iterable[str] | None = None,
) -> pd.DataFrame:
    """Copy the sample and optionally align predictions using unique (possibly composite) IDs.

    Without IDs, predictions must already be in sample order. Prefer explicit IDs for inference.
    Alignment rejects missing, duplicate, extra, or absent keys rather than guessing row order.
    """
    submission = sample.copy()
    ids = [id_columns] if isinstance(id_columns, str) else list(id_columns or [])
    order = None
    if prediction_ids is not None:
        if not ids:
            raise ValueError("id_columns is required with prediction_ids")
        if isinstance(prediction_ids, pd.DataFrame):
            keys = prediction_ids[ids].reset_index(drop=True)
        elif len(ids) == 1:
            keys = pd.DataFrame({ids[0]: list(prediction_ids)})
        else:
            raise ValueError("Composite prediction IDs must be a DataFrame")
        for name, frame in (("sample", sample), ("predictions", keys)):
            if (
                frame[ids].isna().any().any()
                or frame[ids].eq("").any().any()
                or frame.duplicated(ids).any()
            ):
                raise ValueError(f"{name} IDs must be nonmissing and unique across {ids}")
        expected = pd.MultiIndex.from_frame(sample[ids])
        actual = pd.MultiIndex.from_frame(keys)
        order = actual.get_indexer(expected)
        if len(actual) != len(expected) or (order < 0).any():
            raise ValueError("Prediction IDs must match the sample IDs exactly")
    for column, values in predictions.items():
        if column not in submission.columns:
            raise KeyError(f"Prediction column not present in sample submission: {column}")
        if len(values) != len(submission):
            raise ValueError(
                f"Column {column} has {len(values)} values; expected {len(submission)}"
            )
        if column in ids:
            raise ValueError(f"Cannot overwrite ID column: {column}")
        values = list(values)
        submission[column] = values if order is None else [values[index] for index in order]
    return submission




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def build_submission(
    sample: pd.DataFrame,
    predictions: Mapping[str, Sequence[object]],
    *,
    prediction_ids: pd.DataFrame | Sequence[object] | None = None,
    id_columns: str | Iterable[str] | None = None,
) -&gt; pd.DataFrame:
    &quot;&quot;&quot;Copy the sample and optionally align predictions using unique (possibly composite) IDs.

    Without IDs, predictions must already be in sample order. Prefer explicit IDs for inference.
    Alignment rejects missing, duplicate, extra, or absent keys rather than guessing row order.
    &quot;&quot;&quot;
    submission = sample.copy()
    ids = [id_columns] if isinstance(id_columns, str) else list(id_columns or [])
    order = None
    if prediction_ids is not None:
        if not ids:
            raise ValueError(&quot;id_columns is required with prediction_ids&quot;)
        if isinstance(prediction_ids, pd.DataFrame):
            keys = prediction_ids[ids].reset_index(drop=True)
        elif len(ids) == 1:
            keys = pd.DataFrame({ids[0]: list(prediction_ids)})
        else:
            raise ValueError(&quot;Composite prediction IDs must be a DataFrame&quot;)
        for name, frame in ((&quot;sample&quot;, sample), (&quot;predictions&quot;, keys)):
            if (
                frame[ids].isna().any().any()
                or frame[ids].eq(&quot;&quot;).any().any()
                or frame.duplicated(ids).any()
            ):
                raise ValueError(f&quot;{name} IDs must be nonmissing and unique across {ids}&quot;)
        expected = pd.MultiIndex.from_frame(sample[ids])
        actual = pd.MultiIndex.from_frame(keys)
        order = actual.get_indexer(expected)
        if len(actual) != len(expected) or (order &lt; 0).any():
            raise ValueError(&quot;Prediction IDs must match the sample IDs exactly&quot;)
    for column, values in predictions.items():
        if column not in submission.columns:
            raise KeyError(f&quot;Prediction column not present in sample submission: {column}&quot;)
        if len(values) != len(submission):
            raise ValueError(
                f&quot;Column {column} has {len(values)} values; expected {len(submission)}&quot;
            )
        if column in ids:
            raise ValueError(f&quot;Cannot overwrite ID column: {column}&quot;)
        values = list(values)
        submission[column] = values if order is None else [values[index] for index in order]
    return submission


</code></pre>
</details>


### Supplied helper — `validate_submission`

Check schema, row count, ID order, missing values, and optional allowed labels.




In [ ]:
def validate_submission(
    submission: pd.DataFrame,
    sample: pd.DataFrame,
    *,
    id_columns: str | Iterable[str] | None = None,
    allowed_labels: Mapping[str, Iterable[object]] | None = None,
) -> SubmissionValidation:
    """Check schema, row count, ID order, missing values, and optional allowed labels."""
    errors: list[str] = []
    warnings: list[str] = []
    if submission.columns.duplicated().any() or sample.columns.duplicated().any():
        errors.append("Submission and sample column names must be unique")
        return SubmissionValidation(valid=False, errors=errors)
    if list(submission.columns) != list(sample.columns):
        errors.append(
            f"Columns/order differ: got {list(submission.columns)}, expected {list(sample.columns)}"
        )
    if len(submission) != len(sample):
        errors.append(f"Row count differs: got {len(submission)}, expected {len(sample)}")
    if submission.isna().any().any():
        columns = submission.columns[submission.isna().any()].tolist()
        errors.append(f"Missing values in columns: {columns}")
    numeric = submission.select_dtypes(include="number")
    if not np.isfinite(numeric.to_numpy(dtype=float, na_value=np.nan)).all():
        errors.append("Non-finite numeric values are not allowed")
    ids = [id_columns] if isinstance(id_columns, str) else list(id_columns or [])
    for column in ids:
        if column not in submission or column not in sample:
            errors.append(f"Missing ID column: {column}")
        elif submission[column].isna().any() or submission[column].eq("").any():
            errors.append(f"Missing ID values in column: {column}")
        elif len(submission) == len(sample) and not submission[column].equals(sample[column]):
            errors.append(f"ID values/order changed in column: {column}")
    if ids and all(column in submission for column in ids) and submission.duplicated(ids).any():
        warnings.append(f"Duplicate ID keys across columns: {ids}")
    for column, labels in (allowed_labels or {}).items():
        if column not in submission:
            errors.append(f"Missing allowed-label column: {column}")
        else:
            invalid = set(submission[column].dropna().unique()) - set(labels)
            if invalid:
                errors.append(f"Unexpected labels in {column}: {sorted(map(str, invalid))[:10]}")
    return SubmissionValidation(valid=not errors, errors=errors, warnings=warnings)




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def validate_submission(
    submission: pd.DataFrame,
    sample: pd.DataFrame,
    *,
    id_columns: str | Iterable[str] | None = None,
    allowed_labels: Mapping[str, Iterable[object]] | None = None,
) -&gt; SubmissionValidation:
    &quot;&quot;&quot;Check schema, row count, ID order, missing values, and optional allowed labels.&quot;&quot;&quot;
    errors: list[str] = []
    warnings: list[str] = []
    if submission.columns.duplicated().any() or sample.columns.duplicated().any():
        errors.append(&quot;Submission and sample column names must be unique&quot;)
        return SubmissionValidation(valid=False, errors=errors)
    if list(submission.columns) != list(sample.columns):
        errors.append(
            f&quot;Columns/order differ: got {list(submission.columns)}, expected {list(sample.columns)}&quot;
        )
    if len(submission) != len(sample):
        errors.append(f&quot;Row count differs: got {len(submission)}, expected {len(sample)}&quot;)
    if submission.isna().any().any():
        columns = submission.columns[submission.isna().any()].tolist()
        errors.append(f&quot;Missing values in columns: {columns}&quot;)
    numeric = submission.select_dtypes(include=&quot;number&quot;)
    if not np.isfinite(numeric.to_numpy(dtype=float, na_value=np.nan)).all():
        errors.append(&quot;Non-finite numeric values are not allowed&quot;)
    ids = [id_columns] if isinstance(id_columns, str) else list(id_columns or [])
    for column in ids:
        if column not in submission or column not in sample:
            errors.append(f&quot;Missing ID column: {column}&quot;)
        elif submission[column].isna().any() or submission[column].eq(&quot;&quot;).any():
            errors.append(f&quot;Missing ID values in column: {column}&quot;)
        elif len(submission) == len(sample) and not submission[column].equals(sample[column]):
            errors.append(f&quot;ID values/order changed in column: {column}&quot;)
    if ids and all(column in submission for column in ids) and submission.duplicated(ids).any():
        warnings.append(f&quot;Duplicate ID keys across columns: {ids}&quot;)
    for column, labels in (allowed_labels or {}).items():
        if column not in submission:
            errors.append(f&quot;Missing allowed-label column: {column}&quot;)
        else:
            invalid = set(submission[column].dropna().unique()) - set(labels)
            if invalid:
                errors.append(f&quot;Unexpected labels in {column}: {sorted(map(str, invalid))[:10]}&quot;)
    return SubmissionValidation(valid=not errors, errors=errors, warnings=warnings)


</code></pre>
</details>


### Supplied helper — `write_submission`

Validate then atomically write a UTF-8 CSV submission.




In [ ]:
def write_submission(
    submission: pd.DataFrame,
    path: Path | str,
    *,
    sample: pd.DataFrame | None = None,
    id_columns: str | Iterable[str] | None = None,
    allowed_labels: Mapping[str, Iterable[object]] | None = None,
) -> Path:
    """Validate then atomically write a UTF-8 CSV submission."""
    if sample is not None:
        validation = validate_submission(
            submission, sample, id_columns=id_columns, allowed_labels=allowed_labels
        )
        validation.raise_for_errors()
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f".{destination.name}.{uuid4().hex}.tmp")
    try:
        submission.to_csv(temporary, index=False, encoding="utf-8")
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def write_submission(
    submission: pd.DataFrame,
    path: Path | str,
    *,
    sample: pd.DataFrame | None = None,
    id_columns: str | Iterable[str] | None = None,
    allowed_labels: Mapping[str, Iterable[object]] | None = None,
) -&gt; Path:
    &quot;&quot;&quot;Validate then atomically write a UTF-8 CSV submission.&quot;&quot;&quot;
    if sample is not None:
        validation = validate_submission(
            submission, sample, id_columns=id_columns, allowed_labels=allowed_labels
        )
        validation.raise_for_errors()
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f&quot;.{destination.name}.{uuid4().hex}.tmp&quot;)
    try:
        submission.to_csv(temporary, index=False, encoding=&quot;utf-8&quot;)
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination


</code></pre>
</details>


### Supplied helper — `_mapping`

Run this supplied definition before its callers. Its inputs and invariants are explained in this
section.




In [ ]:
def _mapping(value: Any) -> Mapping[str, Any]:
    if is_dataclass(value):
        return asdict(value)
    if isinstance(value, Mapping):
        return value
    raise TypeError("Expected a dataclass or mapping")




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def _mapping(value: Any) -&gt; Mapping[str, Any]:
    if is_dataclass(value):
        return asdict(value)
    if isinstance(value, Mapping):
        return value
    raise TypeError(&quot;Expected a dataclass or mapping&quot;)


</code></pre>
</details>


### Supplied helper — `technical_report`

Render the standard competition handoff report as Markdown text.




In [ ]:
def technical_report(
    *,
    task: str,
    model: str,
    data_summary: Mapping[str, Any],
    split_summary: Mapping[str, Any],
    training_config: Any,
    validation_metrics: Mapping[str, float],
    augmentations: list[str] | None = None,
    checkpoint: str | None = None,
    notes: list[str] | None = None,
) -> str:
    """Render the standard competition handoff report as Markdown text."""
    config = _mapping(training_config)
    lines = [
        "# Technical Report",
        "",
        f"- Task: {task}",
        f"- Model: {model}",
        f"- Checkpoint: {checkpoint or 'not recorded'}",
        "",
        "## Data and validation",
        "",
    ]
    lines.extend(f"- {key}: {value}" for key, value in data_summary.items())
    lines.extend(f"- Split {key}: {value}" for key, value in split_summary.items())
    lines.extend(["", "## Training", ""])
    lines.extend(f"- {key}: {value}" for key, value in config.items())
    lines.extend(["", "## Validation metrics", ""])
    lines.extend(f"- {key}: {value:.6f}" for key, value in validation_metrics.items())
    lines.extend(["", "## Augmentations", ""])
    lines.extend(f"- {item}" for item in (augmentations or ["None"]))
    if notes:
        lines.extend(["", "## Notes", ""])
        lines.extend(f"- {item}" for item in notes)
    return "\n".join(lines) + "\n"




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def technical_report(
    *,
    task: str,
    model: str,
    data_summary: Mapping[str, Any],
    split_summary: Mapping[str, Any],
    training_config: Any,
    validation_metrics: Mapping[str, float],
    augmentations: list[str] | None = None,
    checkpoint: str | None = None,
    notes: list[str] | None = None,
) -&gt; str:
    &quot;&quot;&quot;Render the standard competition handoff report as Markdown text.&quot;&quot;&quot;
    config = _mapping(training_config)
    lines = [
        &quot;# Technical Report&quot;,
        &quot;&quot;,
        f&quot;- Task: {task}&quot;,
        f&quot;- Model: {model}&quot;,
        f&quot;- Checkpoint: {checkpoint or &#x27;not recorded&#x27;}&quot;,
        &quot;&quot;,
        &quot;## Data and validation&quot;,
        &quot;&quot;,
    ]
    lines.extend(f&quot;- {key}: {value}&quot; for key, value in data_summary.items())
    lines.extend(f&quot;- Split {key}: {value}&quot; for key, value in split_summary.items())
    lines.extend([&quot;&quot;, &quot;## Training&quot;, &quot;&quot;])
    lines.extend(f&quot;- {key}: {value}&quot; for key, value in config.items())
    lines.extend([&quot;&quot;, &quot;## Validation metrics&quot;, &quot;&quot;])
    lines.extend(f&quot;- {key}: {value:.6f}&quot; for key, value in validation_metrics.items())
    lines.extend([&quot;&quot;, &quot;## Augmentations&quot;, &quot;&quot;])
    lines.extend(f&quot;- {item}&quot; for item in (augmentations or [&quot;None&quot;]))
    if notes:
        lines.extend([&quot;&quot;, &quot;## Notes&quot;, &quot;&quot;])
        lines.extend(f&quot;- {item}&quot; for item in notes)
    return &quot;\n&quot;.join(lines) + &quot;\n&quot;


</code></pre>
</details>


### Supplied helper — `write_technical_report`

Write a UTF-8 technical report, creating its parent directory.




In [ ]:
def write_technical_report(content: str, path: Path | str) -> Path:
    """Write a UTF-8 technical report, creating its parent directory."""
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding="utf-8")
    return destination




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def write_technical_report(content: str, path: Path | str) -&gt; Path:
    &quot;&quot;&quot;Write a UTF-8 technical report, creating its parent directory.&quot;&quot;&quot;
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding=&quot;utf-8&quot;)
    return destination


</code></pre>
</details>


### Supplied helper — `record_run`

Save configuration/fold positions and a matching human-readable technical report.




In [ ]:
def record_run(
    output_dir: Path | str,
    *,
    task: str,
    model: str,
    configuration: Any,
    validation_metrics: Mapping[str, float],
    split: SplitResult | None = None,
    data_summary: Mapping[str, Any] | None = None,
    inference_options: Mapping[str, Any] | None = None,
    checkpoint: str | None = None,
    notes: list[str] | None = None,
) -> Path:
    """Save configuration/fold positions and a matching human-readable technical report."""
    root = Path(output_dir)
    split_summary = (
        {"train_rows": len(split.train), "valid_rows": len(split.valid)} if split else {}
    )
    payload = {
        "task": task,
        "model": model,
        "configuration": _mapping(configuration),
        "validation_metrics": dict(validation_metrics),
        "data_summary": dict(data_summary or {}),
        "inference_options": dict(inference_options or {}),
        "checkpoint": checkpoint,
        "notes": notes or [],
        "split": {
            **split_summary,
            "train_indices": split.train_indices if split else [],
            "valid_indices": split.valid_indices if split else [],
        },
    }
    save_json(payload, root / "run.json")
    report = technical_report(
        task=task,
        model=model,
        data_summary=data_summary or {},
        split_summary=split_summary,
        training_config=configuration,
        validation_metrics=validation_metrics,
        checkpoint=checkpoint,
        notes=[*(notes or []), f"Inference options: {dict(inference_options or {})}"],
    )
    return write_technical_report(report, root / "technical_report.md")




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def record_run(
    output_dir: Path | str,
    *,
    task: str,
    model: str,
    configuration: Any,
    validation_metrics: Mapping[str, float],
    split: SplitResult | None = None,
    data_summary: Mapping[str, Any] | None = None,
    inference_options: Mapping[str, Any] | None = None,
    checkpoint: str | None = None,
    notes: list[str] | None = None,
) -&gt; Path:
    &quot;&quot;&quot;Save configuration/fold positions and a matching human-readable technical report.&quot;&quot;&quot;
    root = Path(output_dir)
    split_summary = (
        {&quot;train_rows&quot;: len(split.train), &quot;valid_rows&quot;: len(split.valid)} if split else {}
    )
    payload = {
        &quot;task&quot;: task,
        &quot;model&quot;: model,
        &quot;configuration&quot;: _mapping(configuration),
        &quot;validation_metrics&quot;: dict(validation_metrics),
        &quot;data_summary&quot;: dict(data_summary or {}),
        &quot;inference_options&quot;: dict(inference_options or {}),
        &quot;checkpoint&quot;: checkpoint,
        &quot;notes&quot;: notes or [],
        &quot;split&quot;: {
            **split_summary,
            &quot;train_indices&quot;: split.train_indices if split else [],
            &quot;valid_indices&quot;: split.valid_indices if split else [],
        },
    }
    save_json(payload, root / &quot;run.json&quot;)
    report = technical_report(
        task=task,
        model=model,
        data_summary=data_summary or {},
        split_summary=split_summary,
        training_config=configuration,
        validation_metrics=validation_metrics,
        checkpoint=checkpoint,
        notes=[*(notes or []), f&quot;Inference options: {dict(inference_options or {})}&quot;],
    )
    return write_technical_report(report, root / &quot;technical_report.md&quot;)


</code></pre>
</details>


### Exercise 8 — `deliver_submission`

Input: sample template, named predictions, prediction IDs, key-column name and output path.
Return: ID-aligned checked frame and saved CSV path. Reject duplicated, missing or extra
prediction keys.

<details>
<summary>Hint 1 — choose the operation</summary>

One image may contain many boxes, so annotation rows must first be grouped by image_id. The
output is a prediction string of confidence and xywh boxes per image, including a valid empty
string when no box survives.

</details>

<details>
<summary>Hint 2 — implementation direction</summary>

Build by ID, raise validation errors, then write the validated frame. Never assume test order
equals sample order.

</details>




In [ ]:
def deliver_submission(
    sample_frame: pd.DataFrame,
    predictions: Mapping[str, Sequence[object]],
    prediction_ids: Sequence[object],
    id_column: str,
    output_path: Path,
) -> Any:
    """Input: sample template, named predictions, prediction IDs, key-column name and output
    path. Return: ID-aligned checked frame and saved CSV path. Reject duplicated, missing or
    extra prediction keys.
    """
    raise NotImplementedError('TODO: implement deliver_submission using the stated contract')
    return submission, destination




<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>def deliver_submission(
    sample_frame: pd.DataFrame,
    predictions: Mapping[str, Sequence[object]],
    prediction_ids: Sequence[object],
    id_column: str,
    output_path: Path,
) -&gt; Any:
    &quot;&quot;&quot;Input: sample template, named predictions, prediction IDs, key-column name and output
    path. Return: ID-aligned checked frame and saved CSV path. Reject duplicated, missing or
    extra prediction keys.
    &quot;&quot;&quot;
    submission = build_submission(
        sample_frame, predictions, prediction_ids=prediction_ids, id_columns=id_column
    )
    validate_submission(submission, sample_frame, id_columns=id_column).raise_for_errors()
    destination = write_submission(submission, output_path)
    return submission, destination


</code></pre>
</details>


**Check before continuing.** Predict the result, run the small test below, and use its assertion
message to locate an error. Expected: all assertions pass and the printed success message
appears. These checks do not fit the real candidate.



In [ ]:
toy_sample = pd.DataFrame({"key": ["001", "NA"], "prediction": [0.0, 0.0]})
toy_submission, toy_path = deliver_submission(
    toy_sample, {"prediction": [0.8, 0.2]}, ["NA", "001"], "key", OUTPUT / "checks" / "delivery.csv"
)
assert toy_submission["key"].tolist() == ["001", "NA"], "Preserve literal IDs and sample order"
np.testing.assert_allclose(toy_submission["prediction"], [0.2, 0.8])
toy_loaded = pd.read_csv(toy_path, converters={"key": str}, keep_default_na=False)
assert toy_loaded["key"].tolist() == ["001", "NA"]
try:
    deliver_submission(
        toy_sample,
        {"prediction": [0.2, 0.8]},
        ["001", "001"],
        "key",
        OUTPUT / "checks" / "invalid.csv",
    )
except ValueError:
    pass
else:
    raise AssertionError("Duplicate prediction IDs must be rejected, not guessed")
print("Delivery checks passed: shuffled/literal IDs and duplicate rejection.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>toy_sample = pd.DataFrame({&quot;key&quot;: [&quot;001&quot;, &quot;NA&quot;], &quot;prediction&quot;: [0.0, 0.0]})
toy_submission, toy_path = deliver_submission(
    toy_sample, {&quot;prediction&quot;: [0.8, 0.2]}, [&quot;NA&quot;, &quot;001&quot;], &quot;key&quot;, OUTPUT / &quot;checks&quot; / &quot;delivery.csv&quot;
)
assert toy_submission[&quot;key&quot;].tolist() == [&quot;001&quot;, &quot;NA&quot;], &quot;Preserve literal IDs and sample order&quot;
np.testing.assert_allclose(toy_submission[&quot;prediction&quot;], [0.2, 0.8])
toy_loaded = pd.read_csv(toy_path, converters={&quot;key&quot;: str}, keep_default_na=False)
assert toy_loaded[&quot;key&quot;].tolist() == [&quot;001&quot;, &quot;NA&quot;]
try:
    deliver_submission(
        toy_sample,
        {&quot;prediction&quot;: [0.2, 0.8]},
        [&quot;001&quot;, &quot;001&quot;],
        &quot;key&quot;,
        OUTPUT / &quot;checks&quot; / &quot;invalid.csv&quot;,
    )
except ValueError:
    pass
else:
    raise AssertionError(&quot;Duplicate prediction IDs must be rejected, not guessed&quot;)
print(&quot;Delivery checks passed: shuffled/literal IDs and duplicate rejection.&quot;)

</code></pre>
</details>


In [ ]:
submission, destination = deliver_submission(
    sample, output_values, test[ID_COLUMN], ID_COLUMN, OUTPUT / "submission.csv"
)
roundtrip = pd.read_csv(
    destination, converters={ID_COLUMN: str}, keep_default_na=False, float_precision="round_trip"
)
assert roundtrip[ID_COLUMN].tolist() == sample[ID_COLUMN].tolist()
assert roundtrip.columns.tolist() == sample.columns.tolist()
assert len(roundtrip) == len(sample)
pd.testing.assert_frame_equal(roundtrip.astype(str), submission.astype(str), check_dtype=False)
save_json(
    {"train_ids": tr[ID_COLUMN].tolist(), "validation_ids": va[ID_COLUMN].tolist()},
    OUTPUT / "fold_ids.json",
)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>submission, destination = deliver_submission(
    sample, output_values, test[ID_COLUMN], ID_COLUMN, OUTPUT / &quot;submission.csv&quot;
)
roundtrip = pd.read_csv(
    destination, converters={ID_COLUMN: str}, keep_default_na=False, float_precision=&quot;round_trip&quot;
)
assert roundtrip[ID_COLUMN].tolist() == sample[ID_COLUMN].tolist()
assert roundtrip.columns.tolist() == sample.columns.tolist()
assert len(roundtrip) == len(sample)
pd.testing.assert_frame_equal(roundtrip.astype(str), submission.astype(str), check_dtype=False)
save_json(
    {&quot;train_ids&quot;: tr[ID_COLUMN].tolist(), &quot;validation_ids&quot;: va[ID_COLUMN].tolist()},
    OUTPUT / &quot;fold_ids.json&quot;,
)

</code></pre>
</details>


In [ ]:
save_json(
    {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "seed": SEED,
        "max_rows": MAX_ROWS,
        "image_size": IMAGE_SIZE,
        "model_name": MODEL_NAME,
        "imbalance_weights": USE_IMBALANCE_WEIGHTS,
        "pretrained_allowed": PRETRAINED_ALLOWED,
        "initial_weight_path": WEIGHT_PATH,
        "asset_provenance": ASSET_PROVENANCE,
        "trainer": globals().get("training"),
    },
    OUTPUT / "lesson_config.json",
)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>save_json(
    {
        &quot;epochs&quot;: EPOCHS,
        &quot;batch_size&quot;: BATCH_SIZE,
        &quot;seed&quot;: SEED,
        &quot;max_rows&quot;: MAX_ROWS,
        &quot;image_size&quot;: IMAGE_SIZE,
        &quot;model_name&quot;: MODEL_NAME,
        &quot;imbalance_weights&quot;: USE_IMBALANCE_WEIGHTS,
        &quot;pretrained_allowed&quot;: PRETRAINED_ALLOWED,
        &quot;initial_weight_path&quot;: WEIGHT_PATH,
        &quot;asset_provenance&quot;: ASSET_PROVENANCE,
        &quot;trainer&quot;: globals().get(&quot;training&quot;),
    },
    OUTPUT / &quot;lesson_config.json&quot;,
)

</code></pre>
</details>


In [ ]:
record_run(
    OUTPUT,
    task=FAMILY,
    model=MODEL_DESCRIPTION,
    configuration=competition.to_dict(),
    validation_metrics={
        "official_score": float(validation_score),
        "fit_seconds": fit_seconds,
        "inference_seconds": inference_seconds,
    },
    split=split,
    inference_options={
        "threshold": float(selected_threshold),
        "label_order": TARGET_COLUMNS,
        "image_size": IMAGE_SIZE,
        "model_name": MODEL_NAME,
        "asset_provenance": ASSET_PROVENANCE,
    },
    checkpoint=str(checkpoint_path),
    notes=[
        "Educational candidate, not a reproduced Kaggle leaderboard score.",
        "No test labels, test pseudo-labeling or unauthorized external data used.",
        f"Imbalance weights enabled: {USE_IMBALANCE_WEIGHTS}",
    ],
)



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>record_run(
    OUTPUT,
    task=FAMILY,
    model=MODEL_DESCRIPTION,
    configuration=competition.to_dict(),
    validation_metrics={
        &quot;official_score&quot;: float(validation_score),
        &quot;fit_seconds&quot;: fit_seconds,
        &quot;inference_seconds&quot;: inference_seconds,
    },
    split=split,
    inference_options={
        &quot;threshold&quot;: float(selected_threshold),
        &quot;label_order&quot;: TARGET_COLUMNS,
        &quot;image_size&quot;: IMAGE_SIZE,
        &quot;model_name&quot;: MODEL_NAME,
        &quot;asset_provenance&quot;: ASSET_PROVENANCE,
    },
    checkpoint=str(checkpoint_path),
    notes=[
        &quot;Educational candidate, not a reproduced Kaggle leaderboard score.&quot;,
        &quot;No test labels, test pseudo-labeling or unauthorized external data used.&quot;,
        f&quot;Imbalance weights enabled: {USE_IMBALANCE_WEIGHTS}&quot;,
    ],
)

</code></pre>
</details>


In [ ]:
pd.DataFrame(
    [
        {
            "run": RUN_ID,
            "family": FAMILY,
            "variant": VARIANT,
            "score": validation_score,
            "fit_seconds": fit_seconds,
            "inference_seconds": inference_seconds,
        }
    ]
).to_csv(OUTPUT / "comparison_row.csv", index=False)
copied = sync_artifacts(OUTPUT, DURABLE_OUTPUT)
print("Checked local CSV:", destination, "Durable files copied:", len(copied))



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>pd.DataFrame(
    [
        {
            &quot;run&quot;: RUN_ID,
            &quot;family&quot;: FAMILY,
            &quot;variant&quot;: VARIANT,
            &quot;score&quot;: validation_score,
            &quot;fit_seconds&quot;: fit_seconds,
            &quot;inference_seconds&quot;: inference_seconds,
        }
    ]
).to_csv(OUTPUT / &quot;comparison_row.csv&quot;, index=False)
copied = sync_artifacts(OUTPUT, DURABLE_OUTPUT)
print(&quot;Checked local CSV:&quot;, destination, &quot;Durable files copied:&quot;, len(copied))

</code></pre>
</details>


In [ ]:
assert destination.is_file()
assert submission[ID_COLUMN].tolist() == sample[ID_COLUMN].tolist()
print("Delivery complete. Next: restart inference, then attempt the reconstruction task.")



<details>
<summary>Reveal expected code — compare after your attempt</summary>

<pre><code>assert destination.is_file()
assert submission[ID_COLUMN].tolist() == sample[ID_COLUMN].tolist()
print(&quot;Delivery complete. Next: restart inference, then attempt the reconstruction task.&quot;)

</code></pre>
</details>


### Stop and explain

Which ID-order or encoding mistake would survive a plausible-looking preview? What did the
on-disk round trip prove?

<details>
<summary>Reasoning checkpoint — open after writing your answer</summary>

Shuffled sample IDs and literal strings such as 001/NA require explicit alignment and CSV
parsing. Readback checks the delivered file's schema/order/values; official task semantics and
upload acceptance still need separate confirmation.

</details>



## Transfer and closed-book reconstruction

A new table contains ten annotations per image and a source column. Demonstrate source-disjoint
image splits, a batch with different box counts, and a valid empty prediction string before
fitting.

1. Close the reference notebook and create an empty personal notebook.
2. Rebuild the data contract, split, features/model, fit, official metric and submission path.
3. Deliver a checked small-pack CSV within 45–60 minutes, then reload it in a fresh session.
4. Record elapsed time, hints, bugs and the first function you could not reconstruct.
5. Rebuild that function the next day with a different schema or seed.

For the 2k assistant, send one function signature, input/output shapes and one failing toy test.
Ask for an explanation or correction of that function only. Keep the rest of the notebook
outside the prompt.

### Controlled comparison

Compare MobileNet and ResNet-FPN using the same image/source split and grader. Include full test
inference time. Test resolution or confidence independently; do not introduce an ensemble before
measuring its delivery cost.

A first correct submission is the readiness gate. Advanced complexity is selected only after an
honest metric/runtime comparison; no test fitting or model-weight permission is implied.
